In [1]:
# ============================================================
# PIPELINE ANOMALI: "NILAI DI LUAR RENTANG PER-MESIN"
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
from typing import Dict, Tuple, Optional, List

# ---------- Utils ----------
def _coerce_num(x):
    if isinstance(x, str):
        x = x.replace(",", ".")
    return pd.to_numeric(x, errors="coerce")

def _ensure_dir(p):
    Path(p).parent.mkdir(parents=True, exist_ok=True)

# ---------- 1) Load thresholds (min/max) per machine ----------
def load_thresholds(path: str, sheet: Optional[str] = None) -> pd.DataFrame:
    """
    Path bisa .xlsx/.xls/.csv. Sheet opsional untuk Excel.
    Harus ada kolom: (case-insensitive, bebas spasi)
      - tag number / machine / mesin
      - maximum / max
      - minimum / min
    """
    if path.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(path, sheet_name=sheet) if sheet else pd.read_excel(path)
    else:
        df = pd.read_csv(path)
    # normalisasi header
    def norm(s): return (str(s) or "").strip().lower().replace(" ", "").replace("_","")
    cols = {c: norm(c) for c in df.columns}
    # map kolom
    col_tag = None
    for c,n in cols.items():
        if n in {"tagnumber","tagnumber.","tag","machine","mesin"}:
            col_tag = c; break
    col_max = None
    for c,n in cols.items():
        if "max" in n: col_max = c; break
    col_min = None
    for c,n in cols.items():
        if "min" in n: col_min = c; break
    if not (col_tag and col_min and col_max):
        raise ValueError(f"Kolom wajib tidak ditemukan. Ada: {list(df.columns)}")

    out = df[[col_tag, col_max, col_min]].copy()
    out.columns = ["machine_id", "max_value", "min_value"]
    out["machine_id"] = out["machine_id"].astype(str).str.strip()
    out["max_value"] = out["max_value"].apply(_coerce_num)
    out["min_value"] = out["min_value"].apply(_coerce_num)

    # bersihkan & swap kalau min > max
    out = out.dropna(subset=["machine_id","max_value","min_value"]).reset_index(drop=True)
    swap_mask = out["min_value"] > out["max_value"]
    if swap_mask.any():
        tmp = out.loc[swap_mask, "min_value"].copy()
        out.loc[swap_mask, "min_value"] = out.loc[swap_mask, "max_value"]
        out.loc[swap_mask, "max_value"] = tmp
    return out

def build_threshold_map(df_thresh: pd.DataFrame) -> Dict[str, Tuple[float,float]]:
    return {row["machine_id"]: (float(row["min_value"]), float(row["max_value"])) for _,row in df_thresh.iterrows()}

# ---------- 2) Detect anomalies (outside [min,max]) ----------
def detect_range_anomalies(
    df_raw: pd.DataFrame,
    thresholds: Dict[str, Tuple[float,float]],
    ts_col="ts", machine_col="machine_id", value_col="value",
    start: Optional[str] = None, end: Optional[str] = None,
    machines: Optional[List[str]] = None,
    fallback="infer"   # "skip" → abaikan mesin tanpa threshold; "infer" → pakai quantile 1%-99%
) -> pd.DataFrame:
    """
    Return df dengan kolom tambahan:
      - expected_min, expected_max
      - is_anomaly (bool)
      - which_bound ('below'/'above'/None)
      - distance (nilai keluarannya)
      - distance_pct (relatif terhadap rentang)
    """
    df = df_raw.copy()
    df[ts_col] = pd.to_datetime(df[ts_col], errors="coerce")
    df[value_col] = df[value_col].apply(_coerce_num)
    df = df.dropna(subset=[ts_col, machine_col, value_col])

    if start: df = df[df[ts_col] >= pd.Timestamp(start)]
    if end:   df = df[df[ts_col] <= pd.Timestamp(end)]
    if machines:
        df = df[df[machine_col].isin(machines)]

    if df.empty:
        return df.assign(expected_min=np.nan, expected_max=np.nan,
                         is_anomaly=False, which_bound=None,
                         distance=np.nan, distance_pct=np.nan)

    # siapkan expected_min/expected_max per baris
    mins = []; maxs = []
    for mid,val in zip(df[machine_col].astype(str), df[value_col].astype(float)):
        if mid in thresholds:
            vmin, vmax = thresholds[mid]
        else:
            if fallback == "skip":
                vmin, vmax = np.nan, np.nan
            else:
                # infer dari distribusi mesin itu di range terpilih
                s = df.loc[df[machine_col]==mid, value_col].astype(float)
                if len(s) >= 10:
                    vmin, vmax = float(s.quantile(0.01)), float(s.quantile(0.99))
                else:
                    vmin, vmax = float(s.min()), float(s.max())
        mins.append(vmin); maxs.append(vmax)
    df["expected_min"] = mins
    df["expected_max"] = maxs

    # flag anomaly
    df["is_anomaly"] = (df[value_col] < df["expected_min"]) | (df[value_col] > df["expected_max"])
    df["which_bound"] = np.where(df[value_col] < df["expected_min"], "below",
                          np.where(df[value_col] > df["expected_max"], "above", None))
    # distance absolut ke batas terdekat
    df["distance"] = np.where(
        df["which_bound"]=="below", df["expected_min"] - df[value_col],
        np.where(df["which_bound"]=="above", df[value_col] - df["expected_max"], 0.0)
    )
    # distance relatif terhadap lebar rentang; kalau rentang 0 → pakai |val - bound|
    width = (df["expected_max"] - df["expected_min"]).replace(0, np.nan)
    df["distance_pct"] = np.where(
        df["is_anomaly"],
        df["distance"] / width.replace({0: np.nan}),
        0.0
    )
    df["distance_pct"] = df["distance_pct"].fillna(df["distance"].abs())
    return df

# ---------- 3) Summaries per machine ----------
def summarize_anomalies(df_anom: pd.DataFrame, ts_col="ts", machine_col="machine_id", value_col="value", top_k=5):
    if df_anom.empty:
        return {"machines": {}, "combined": {"total": 0}}
    res = {}
    for mid, g in df_anom.groupby(machine_col):
        total = int(g["is_anomaly"].sum())
        if total == 0:
            res[mid] = {"total": 0}
            continue
        g1 = g[g["is_anomaly"]].sort_values(ts_col)
        earliest = g1[ts_col].iloc[0].isoformat()
        latest   = g1[ts_col].iloc[-1].isoformat()
        # contoh anomali terparah (by distance_pct, descending)
        worst = g1.sort_values("distance_pct", ascending=False).head(top_k)[
            [ts_col, value_col, "which_bound", "expected_min", "expected_max", "distance", "distance_pct"]
        ].copy()
        worst[ts_col] = worst[ts_col].astype("datetime64[ns]").dt.strftime("%Y-%m-%d %H:%M:%S")
        res[mid] = {
            "total": total,
            "earliest": earliest,
            "latest": latest,
            "worst_examples": worst.to_dict(orient="records")
        }
    combined = {"total": int(df_anom["is_anomaly"].sum())}
    return {"machines": res, "combined": combined}

# ---------- 4) Plot (highlight anomalies) ----------
def plot_anomaly_series(df_anom, machine_id, *, ts_col="ts", value_col="value",
                        save_path=None, title_suffix="ANOMALY"):
    d = df_anom[df_anom["machine_id"]==machine_id].copy()
    if d.empty: return None
    d = d.sort_values(ts_col)
    # garis normal
    plt.figure()
    plt.plot(d[ts_col], d[value_col], marker=".", linestyle="-")
    # highlight anomaly
    da = d[d["is_anomaly"]]
    if not da.empty:
        plt.scatter(da[ts_col], da[value_col], s=30)  # default color; policy: no custom colors
    # overlay batas (opsional: step)
    if d["expected_min"].notna().any():
        plt.plot(d[ts_col], d["expected_min"], linestyle="--")
    if d["expected_max"].notna().any():
        plt.plot(d[ts_col], d["expected_max"], linestyle="--")
    plt.xlabel("Waktu"); plt.ylabel(value_col); plt.title(f"{machine_id} — {title_suffix}")
    plt.tight_layout()
    if save_path:
        _ensure_dir(save_path); plt.savefig(save_path, dpi=140); plt.close(); return save_path
    return None

# ---------- 5) Orkestrator: from intent + subintent ----------
def serve_anomaly_pipeline(
    df_raw: pd.DataFrame,
    thresholds_map: Dict[str, Tuple[float,float]],
    intent: Optional[dict] = None,     # boleh None → pakai semua data
    subintent: str = "summary",        # "anom_only"|"count"|"list"|"plot"|"summary"
    out_dir: str = "./plots_anom",
    ts_col="ts", machine_col="machine_id", value_col="value"
):
    # 1) filter waktu & mesin dari intent (kalau ada)
    start = end = None; machines = None
    if intent:
        tr = intent.get("time_range") or {}
        start, end = tr.get("start"), tr.get("end")
        if (intent.get("scope") == "SUBSET") and intent.get("machines"):
            machines = intent["machines"]

    # 2) deteksi anomali
    df_anom = detect_range_anomalies(
        df_raw, thresholds_map, ts_col=ts_col, machine_col=machine_col, value_col=value_col,
        start=start, end=end, machines=machines, fallback="infer"  # ubah ke "skip" kalau mau strict
    )

    # 3) ringkasan
    summary = summarize_anomalies(df_anom, ts_col=ts_col, machine_col=machine_col, value_col=value_col, top_k=5)

    # 4) efisiensi: kalau user cuma minta jumlah/list → skip plot
    only_counts = subintent in {"count","anom_only","list"}
    plot_paths = {}
    if not only_counts:
        Path(out_dir).mkdir(parents=True, exist_ok=True)
        for mid in df_anom[machine_col].unique():
            p = os.path.join(out_dir, f"{mid}_anomaly.png")
            out = plot_anomaly_series(df_anom, mid, ts_col=ts_col, value_col=value_col,
                                      save_path=p, title_suffix="ANOMALY vs RANGE")
            if out:
                plot_paths[mid] = out

    # 5) metadata ringkas untuk LLM
    meta = {
        "time_used": {"start": start, "end": end},
        "machines": machines or "ALL",
        "threshold_source": "sheet" ,
        "summary": summary
    }
    return {"meta": meta, "plot_paths": plot_paths, "dataframe": df_anom}

# ---------- 6) Pretty text ----------
def pretty_anomaly_text(result, decimal_comma=True, places=6):
    if result is None: return "No result."
    s = result["meta"]["summary"]; lines = []
    def fmt(x, p=6):
        if x is None or (isinstance(x,float) and (np.isnan(x) or np.isinf(x))): return "NA"
        s = f"{x:.{p}f}"
        return s.replace(".", ",") if decimal_comma else s
    lines.append(f"Total anomali: {s['combined']['total']}.")
    for mid, v in s["machines"].items():
        if v.get("total",0) == 0:
            lines.append(f"- [{mid}] tidak ada anomali.")
            continue
        lines.append(f"- [{mid}] anomali={v['total']}, awal={v['earliest'][:19]}, akhir={v['latest'][:19]}.")
        for w in v["worst_examples"]:
            lines.append(
                f"    • {w['ts']} | val={fmt(w['value'],places)} | {w['which_bound']} "
                f"({fmt(w['distance'],places)} dari batas; rentang [{fmt(w['expected_min'],places)}, {fmt(w['expected_max'],places)}])"
            )
    return "\n".join(lines)


In [4]:
# 1) siapkan df_raw (ts datetime, machine_id string, value float)
# df_raw = ...

# 2) load threshold dari sheet (yang ada tabel Tag Number, Max, Min)
# contoh:
# THRESH_PATH = "/mnt/data/original_data.xlsx"
# df_th = load_thresholds(THRESH_PATH, sheet="Sheet1")
# Atau kalau sudah ada dict:
df_raw=pd.read_csv(r'/workspace/df_raw.csv')
df_th = pd.DataFrame({
    "machine_id": ["XP888A","PQ667"],
    "max_value": [10, 140],
    "min_value": [0, 110],
})
th_map = build_threshold_map(df_th)

# 3) (opsional) pakai interpret() untuk ambil rentang waktu & mesin dari query
# pred = runner.predict(q); intent = interpret(q, pred)

# 4) jalankan pipeline
res = serve_anomaly_pipeline(
    df_raw, th_map,
    intent=None,              # atau intent dari interpret()
    subintent="summary",      # "count"/"list" kalau cuma mau jumlah/daftar → lebih efisien
    out_dir="./plots_anom",
    ts_col="ts", machine_col="machine_id", value_col="value"
)

print(pretty_anomaly_text(res))
print("Plots:", res["plot_paths"])   # tampilkan gambar di UI kamu
# Dataframe dengan flag per baris: res["dataframe"].head()


Total anomali: 55.
- [PQ667] anomali=32, awal=2025-01-01T12:15:57, akhir=2025-02-04T08:00:57.
    • 2025-01-13 00:00:57 | val=90,223010 | below (19,776990 dari batas; rentang [110,000000, 140,000000])
    • 2025-01-13 00:03:57 | val=90,223010 | below (19,776990 dari batas; rentang [110,000000, 140,000000])
    • 2025-01-13 00:06:57 | val=95,499737 | below (14,500263 dari batas; rentang [110,000000, 140,000000])
    • 2025-01-18 06:09:57 | val=98,201000 | below (11,799000 dari batas; rentang [110,000000, 140,000000])
    • 2025-01-18 06:12:57 | val=98,201000 | below (11,799000 dari batas; rentang [110,000000, 140,000000])
- [XP888A] anomali=23, awal=2025-01-01T13:18:57, akhir=2025-02-04T23:39:57.
    • 2025-01-17 04:12:57 | val=19,234000 | above (9,234000 dari batas; rentang [0,000000, 10,000000])
    • 2025-01-17 04:09:57 | val=17,876542 | above (7,876542 dari batas; rentang [0,000000, 10,000000])
    • 2025-01-29 19:54:57 | val=17,223141 | above (7,223141 dari batas; rentang [0,000000

In [18]:
# =======================
# PATCH: Time Parser (ID)
# =======================
import re
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import List, Dict, Any, Tuple, Optional
import calendar

# set "hari ini" sesuai kebutuhanmu
NOW = datetime(2025, 10, 23)

ID_MONTHS = {
    "januari":1,"jan":1,"jan.":1,
    "februari":2,"feb":2,"feb.":2,
    "maret":3,"mar":3,"mar.":3,
    "april":4,"apr":4,"apr.":4,
    "mei":5,
    "juni":6,"jun":6,"jun.":6,
    "juli":7,"jul":7,"jul.":7,
    "agustus":8,"agu":8,"agt":8,"agust":8,
    "september":9,"sep":9,"sept":9,"sep.":9,
    "oktober":10,"okt":10,"okt.":10,
    "november":11,"nov":11,"nov.":11,
    "desember":12,"des":12,"des.":12,
}

ID_DAYS = {"senin":0,"selasa":1,"rabu":2,"kamis":3,"jumat":4,"jum’at":4,"sabtu":5,"minggu":6,"ahad":6}
ORDS = {"pertama":1,"ke-1":1,"kesatu":1,"1":1,
        "kedua":2,"ke-2":2,"2":2,
        "ketiga":3,"ke-3":3,"3":3,
        "keempat":4,"ke-4":4,"4":4,
        "kelima":5,"ke-5":5,"5":5}

@dataclass
class TimeInterval:
    start: datetime
    end: datetime
    granularity: str   # HOUR/DAY/WEEK/MONTH/YEAR
    note: str

def start_of_day(d): return d.replace(hour=0, minute=0, second=0, microsecond=0)
def end_of_day(d):   return d.replace(hour=23, minute=59, second=59, microsecond=0)

def month_range(y:int, m:int) -> Tuple[datetime, datetime]:
    last = calendar.monthrange(y, m)[1]
    return datetime(y,m,1,0,0,0), datetime(y,m,last,23,59,59)

def year_range(y:int) -> Tuple[datetime, datetime]:
    return datetime(y,1,1,0,0,0), datetime(y,12,31,23,59,59)

def monday_of(d: datetime) -> datetime:
    return start_of_day(d - timedelta(days=d.weekday()))

def week_range_from_monday(mon: datetime) -> Tuple[datetime, datetime]:
    return mon, end_of_day(mon + timedelta(days=6))

def nth_week_in_month(y:int, m:int, n:int) -> Tuple[datetime, datetime]:
    ms, me = month_range(y, m)
    cur = monday_of(ms)
    weeks = []
    while cur <= me:
        ws, we = week_range_from_monday(cur)
        ws = max(ws, ms); we = min(we, me)
        if ws <= me and we >= ms:
            weeks.append((ws,we))
        cur += timedelta(days=7)
        if len(weeks) >= 6: break
    if not weeks: return ms, me
    n = max(1, min(n, len(weeks)))
    return weeks[n-1]

def merge_intervals(iv: List[TimeInterval]) -> List[TimeInterval]:
    if not iv: return []
    iv = sorted(iv, key=lambda x: x.start)
    out = [iv[0]]
    for it in iv[1:]:
        last = out[-1]
        if it.start <= last.end + timedelta(seconds=1) and it.granularity == last.granularity:
            last.end = max(last.end, it.end)
            last.note = f"{last.note} ∪ {it.note}"
        else:
            out.append(it)
    return out

# ---------- NEW robust date parsers ----------
def _parse_day_month_year_span(text: str) -> Optional[Tuple[datetime, datetime]]:
    """
    Tangkap pola: '10-20 Januari 2025' atau '10 s/d 20 Januari 2025'
    """
    pat = re.compile(
        r"\b(\d{1,2})\s*(?:-|–|sd|s/d|sampai|hingga)\s*(\d{1,2})\s+([A-Za-z\.]+)\s+(\d{4})",
        re.I
    )
    m = pat.search(text)
    if not m: return None
    d1, d2 = int(m.group(1)), int(m.group(2))
    mon = ID_MONTHS.get(m.group(3).lower(), None)
    year = int(m.group(4))
    if not mon: return None
    d1, d2 = sorted([d1, d2])
    s = datetime(year, mon, d1, 0, 0, 0)
    e = datetime(year, mon, d2, 23, 59, 59)
    return s, e

def _parse_day_to_day_month(text: str, default_year: int) -> Optional[Tuple[datetime, datetime]]:
    """
    Tangkap pola: 'tanggal 1 sampai 30 januari' (tahun opsional → pakai default_year)
    """
    pat = re.compile(
        r"(?:tanggal\s*)?(\d{1,2})\s*(?:-|–|sd|s/d|sampai|hingga)\s*(\d{1,2})\s+([A-Za-z\.]+)(?:\s+(\d{4}))?",
        re.I
    )
    m = pat.search(text)
    if not m: return None
    d1, d2 = int(m.group(1)), int(m.group(2))
    mon = ID_MONTHS.get(m.group(3).lower(), None)
    year = int(m.group(4)) if m.group(4) else default_year
    if not mon: return None
    d1, d2 = sorted([d1, d2])
    s = datetime(year, mon, d1, 0, 0, 0)
    e = datetime(year, mon, d2, 23, 59, 59)
    return s, e

def parse_time_complex(text: str, now: datetime = NOW) -> Dict[str, Any]:
    t = text.lower()
    intervals: List[TimeInterval] = []
    notes = []

    # --- 0) Ambil konteks bulan/tahun (kalau disebut) supaya minggu 1–N tepat ---
    month_mentioned = None
    mo = re.search(r"(jan(uari)?|feb(ruari)?|mar(et)?|apr(il)?|mei|jun(i)?|jul(i)?|ag(us?t?us)?|sep(t|tember)?|okt(ober)?|nov(ember)?|des(ember)?)", t)
    if mo: month_mentioned = ID_MONTHS[mo.group(1)]
    year_mentioned = None
    if "tahun lalu" in t:
        year_mentioned = now.year - 1
    elif "tahun ini" in t:
        year_mentioned = now.year
    else:
        yexp = re.search(r"(\d{4})", t)
        if yexp: year_mentioned = int(yexp.group(1))

    # --- 1) Rentang tanggal eksplisit (10–20 Januari 2025 / 1 sampai 30 januari) ---
    dspan = _parse_day_month_year_span(t) or _parse_day_to_day_month(t, year_mentioned or now.year)
    if dspan:
        s,e = dspan
        if e.date() == now.date() and (year_mentioned or month_mentioned) == now.month:
            e = min(e, end_of_day(now))
        intervals.append(TimeInterval(s,e,"DAY","range tanggal"))
        notes.append("date_span")

    # --- 2) Minggu pertama/ke-N bulan X tahun Y (termuat juga 'minggu 1–2', 'minggu 1 dan 2') ---
    week_spans = []
    # a) 1–2
    for m in re.finditer(r"minggu\s+(ke-?\s*\d+|\d+|pertama|kedua|ketiga|keempat|kelima)\s*(?:-|–|sd|s/d|sampai|hingga)\s*(\d+|pertama|kedua|ketiga|keempat|kelima)", t):
        a = ORDS.get(m.group(1).replace(" ",""), None) or ORDS.get(m.group(1), None)
        b = ORDS.get(m.group(2), None)
        if a and b:
            week_spans.append((min(a,b), max(a,b)))
    # b) "1 dan 2"
    m = re.search(r"minggu\s+((?:ke-?\s*\d+|\d+|pertama|kedua|ketiga|keempat|kelima)(?:\s*(?:,|dan)\s*(?:ke-?\s*\d+|\d+|pertama|kedua|ketiga|keempat|kelima))*)", t)
    if m:
        toks = re.split(r"\s*(?:,|dan)\s*", m.group(1))
        nums = []
        for tok in toks:
            nums.append( ORDS.get(tok.replace(" ",""), None) or ORDS.get(tok, None) )
        nums = [n for n in nums if n]
        week_spans += [(n,n) for n in nums]

    if week_spans:
        y = year_mentioned or now.year
        mth = month_mentioned or (now.month if "bulan ini" in t else now.month)
        for a,b in week_spans:
            ws1,we1 = nth_week_in_month(y,mth,a)
            ws2,we2 = nth_week_in_month(y,mth,b)
            s = min(ws1,ws2); e = max(we1,we2)
            intervals.append(TimeInterval(s,e,"WEEK",f"minggu {a}..{b} {mth:02d}-{y}"))
            notes.append("week_span")

    # --- 3) Bulan ini/lalu/spesifik ---
    if "bulan ini" in t:
        s,e = month_range(now.year, now.month)
        e = min(e, end_of_day(now))
        intervals.append(TimeInterval(s,e,"MONTH","bulan ini"))
    if "bulan lalu" in t:
        y,mn = now.year, now.month-1
        if mn==0: y-=1; mn=12
        s,e = month_range(y,mn); intervals.append(TimeInterval(s,e,"MONTH","bulan lalu"))
    # 'Januari 2025' / 'Feb 2025'
    MONTH_YR_RE = re.compile(r"(?P<mon>jan(?:uari)?|feb(?:ruari)?|mar(?:et)?|apr(?:il)?|mei|jun(?:i)?|jul(?:i)?|ag(?:us?t?us)?|sep(?:t|tember)?|okt(?:ober)?|nov(?:ember)?|des(?:ember)?)\s+(?P<year>\d{4})",re.I
    )
    m = MONTH_YR_RE.search(t)
    if m:
        mon_name = m.group("mon").lower()
        yr_text  = m.group("year")
        if yr_text is not None:
            yr = int(yr_text)
            mon = ID_MONTHS[mon_name]
            s, e = month_range(yr, mon)
            if yr == now.year and mon == now.month:
                e = min(e, end_of_day(now))  # MTD
            intervals.append(TimeInterval(s, e, "MONTH", f"bulan {mon:02d}-{yr}"))

    # --- 4) Hari dalam minggu / hari ini / kemarin ---
    for name, wd in ID_DAYS.items():
        if re.search(rf"\b{name}\b", t):
            d = now - timedelta(days=(now.weekday()-wd)%7)
            intervals.append(TimeInterval(start_of_day(d), end_of_day(d), "DAY", f"hari {name}"))
            break
    if "hari ini" in t:
        intervals.append(TimeInterval(start_of_day(now), end_of_day(now), "DAY", "hari ini"))
    if "kemarin" in t:
        d = now - timedelta(days=1)
        intervals.append(TimeInterval(start_of_day(d), end_of_day(d), "DAY", "kemarin"))

    # --- 5) N hari/minggu/bulan terakhir ---
    m = re.search(r"(\d+)\s*(hari|minggu|bulan)\s+terakhir", t)
    if m:
        n = int(m.group(1)); unit = m.group(2)
        if unit=="hari":
            s = start_of_day(now - timedelta(days=n-1)); e = end_of_day(now)
            intervals.append(TimeInterval(s,e,"DAY",f"{n} hari terakhir"))
        elif unit=="minggu":
            s = start_of_day(now - timedelta(days=7*(n-1))); e = end_of_day(now)
            intervals.append(TimeInterval(s,e,"WEEK",f"{n} minggu terakhir"))
        else:
            # mundur (n-1) bulan sampai awal bulan tsb
            y,mon = now.year, now.month-(n-1)
            while mon<=0: y-=1; mon+=12
            s,_ = month_range(y,mon); e = end_of_day(now)
            intervals.append(TimeInterval(s,e,"MONTH",f"{n} bulan terakhir"))

    # --- 6) Tahun ini/lalu/eksplisit ---
    if "tahun ini" in t:
        s,e = year_range(now.year); e = min(e,end_of_day(now))
        intervals.append(TimeInterval(s,e,"YEAR","tahun ini"))
    if "tahun lalu" in t:
        s,e = year_range(now.year-1); intervals.append(TimeInterval(s,e,"YEAR","tahun lalu"))
    yx = re.search(r"tahun\s+(\d{4})", t)
    if yx:
        yy = int(yx.group(1)); s,e = year_range(yy)
        if yy==now.year: e=min(e,end_of_day(now))
        intervals.append(TimeInterval(s,e,"YEAR",f"tahun {yy}"))

    # --- default jika tidak ada apa-apa: bulan ini (MTD) ---
    if not intervals:
        s,e = month_range(now.year, now.month)
        e = min(e, end_of_day(now))
        intervals.append(TimeInterval(s,e,"MONTH","default: bulan ini"))

    # merge + pilih granularity terhalus
    intervals = merge_intervals(intervals)
    order = {"HOUR":0,"DAY":1,"WEEK":2,"MONTH":3,"YEAR":4}
    gran = min(intervals, key=lambda z: order[z.granularity]).granularity

    route = []
    if any(it.granularity=="WEEK" for it in intervals): route.append("slot_week_extractor")
    if any(it.granularity=="MONTH" for it in intervals): route.append("slot_month_extractor")
    if any(it.granularity=="YEAR" for it in intervals):  route.append("slot_year_extractor")
    if any("week_span" in it.note for it in intervals):  route.append("use_week_range_extractor")
    if len(intervals)>1: route.append("merge_intervals")

    return {
        "intervals": [{"start": it.start.isoformat(), "end": it.end.isoformat(),
                       "granularity": it.granularity, "note": it.note} for it in intervals],
        "granularity": gran,
        "route_hints": route
    }

# ====== builder intent (pakai parser baru) ======
MACHINE_PATTERN = re.compile(r"\b([A-Z]{2,}\d{2,}[A-Z0-9]*|[A-Z]{3,}\d+)\b")
def extract_machines(text:str):
    seen=set(); out=[]
    for m in MACHINE_PATTERN.finditer(text.upper()):
        x=m.group(1)
        if x not in seen:
            seen.add(x); out.append(x)
    return out

def build_intent_complex(query_text: str) -> Dict[str, Any]:
    machines = extract_machines(query_text)
    scope = "ALL" if ("semua" in query_text.lower() or "all" in query_text.lower()) else ("SUBSET" if machines else "ALL")
    time_info = parse_time_complex(query_text, NOW)
    return {
        "query": query_text,
        "scope": scope,
        "machines": machines if scope=="SUBSET" else [],
        "granularity": time_info["granularity"],
        "intervals": time_info["intervals"],
        "route_hints": time_info["route_hints"],
        "meta": {"machine_count": len(machines)}
    }

if __name__ == "__main__":
    examples = [
        "Tolong tampilkan mesin XP888A 10–20 Januari 2025, apakah normal?",
        "Tolong tampilkan mesin XP888A dan XR101 2 hari terakhir, apakah terlihat tidak normal?",
        "Apakah ada anomali mesin XP888A dan PQ667 hari ini?",
        "Anomali semua mesin minggu pertama dan kedua Januari 2025",
        "Anomali semua mesin minggu 1 dan 2 bulan ini",
        "Ringkasan semua mesin bulan lalu",
        "XP888A minggu 1-3 Feb 2024",
        "XP888A  tgl 1-3 Feb 2024",
    ]
    for q in examples:
        print("\nQ:", q)
        print(build_intent_complex(q))



Q: Tolong tampilkan mesin XP888A 10–20 Januari 2025, apakah normal?
{'query': 'Tolong tampilkan mesin XP888A 10–20 Januari 2025, apakah normal?', 'scope': 'SUBSET', 'machines': ['XP888A'], 'granularity': 'DAY', 'intervals': [{'start': '2025-01-01T00:00:00', 'end': '2025-01-31T23:59:59', 'granularity': 'MONTH', 'note': 'bulan 01-2025'}, {'start': '2025-01-10T00:00:00', 'end': '2025-01-20T23:59:59', 'granularity': 'DAY', 'note': 'range tanggal'}], 'route_hints': ['slot_month_extractor', 'merge_intervals'], 'meta': {'machine_count': 1}}

Q: Tolong tampilkan mesin XP888A dan XR101 2 hari terakhir, apakah terlihat tidak normal?
{'query': 'Tolong tampilkan mesin XP888A dan XR101 2 hari terakhir, apakah terlihat tidak normal?', 'scope': 'SUBSET', 'machines': ['XP888A', 'XR101'], 'granularity': 'DAY', 'intervals': [{'start': '2025-10-22T00:00:00', 'end': '2025-10-23T23:59:59', 'granularity': 'DAY', 'note': '2 hari terakhir'}], 'route_hints': [], 'meta': {'machine_count': 2}}

Q: Apakah ada an

In [32]:
# =============================================================
# Anomaly — Quad Head Trainer (scope • time_granularity • time_complexity • case)
# Dataset: /mnt/data/anomaly_comprehensive_ds_v2_relclass
# Model base: IndoBERT (bisa diganti "roberta-base")
# =============================================================

# ============ (0) CONFIG ============
DATA_DIR   = "/workspace/anomaly_comprehensive_ds_v2_relclass"  # folder dataset komprehensif + RANGE_REL
OUT_ROOT   = "./runs_anomaly_quad_relclass"                    # folder output
BASE_MODEL = "indobenchmark/indobert-base-p1"                  # opsi lain: "roberta-base"

EPOCHS  = 30
LR      = 3e-5
BATCH   = 16
MAX_LEN = 128
SEED    = 42

import os, json, random, numpy as np, warnings, sys
from pathlib import Path

random.seed(SEED); np.random.seed(SEED)
os.makedirs(OUT_ROOT, exist_ok=True)

# Cek file wajib
REQ = [
    "train_scope.jsonl", "train_time_granularity.jsonl", "train_time_complexity.jsonl", "train_case.jsonl",
    # valid optional (boleh kosong; kita pakai split 12% jika ingin auto buat valid)
    "train_anomaly_comprehensive_relclass_master.jsonl",
]
for f in REQ:
    p = Path(DATA_DIR) / f
    print(f"{f}: {'OK' if p.exists() else 'MISSING'}  -> {p}")

# ============ (1) TRAINER UTILS ============
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch

def compute_metrics_mc(p):
    logits, y = p
    preds = logits.argmax(axis=1)
    return {"acc": accuracy_score(y, preds), "f1_macro": f1_score(y, preds, average="macro")}

def train_multiclass_from_jsonl(model_name, train_jsonl, valid_jsonl=None,
                                out_dir="./out", epochs=6, lr=3e-5, batch=16, max_len=128, seed=42):
    """
    Jika valid_jsonl tidak disediakan atau file tidak ada → auto split (stratified) 12%.
    """
    from sklearn.model_selection import StratifiedShuffleSplit

    # Load train (dan valid kalau ada)
    if valid_jsonl and Path(valid_jsonl).exists():
        ds = load_dataset("json", data_files={"train": train_jsonl, "validation": valid_jsonl})
    else:
        # auto-split
        full = load_dataset("json", data_files={"train": train_jsonl})["train"]
        texts = full["text"]; labels = full["label"]
        idx = np.arange(len(texts))
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.12, random_state=seed)
        tr_idx, va_idx = next(sss.split(idx, labels))
        tr = [{"text": texts[i], "label": labels[i]} for i in tr_idx]
        va = [{"text": texts[i], "label": labels[i]} for i in va_idx]
        # simpan split sementara
        tmp_dir = Path(out_dir) / "_autosplit"; tmp_dir.mkdir(parents=True, exist_ok=True)
        trp = tmp_dir / "train.jsonl"; vap = tmp_dir / "valid.jsonl"
        with open(trp, "w", encoding="utf-8") as f:
            for r in tr: f.write(json.dumps(r, ensure_ascii=False) + "\n")
        with open(vap, "w", encoding="utf-8") as f:
            for r in va: f.write(json.dumps(r, ensure_ascii=False) + "\n")
        ds = load_dataset("json", data_files={"train": str(trp), "validation": str(vap)})

    labels_sorted = sorted(list({ex["label"] for ex in ds["train"]}))
    label2id = {l:i for i,l in enumerate(labels_sorted)}
    id2label = {i:l for l,i in label2id.items()}

    tok = AutoTokenizer.from_pretrained(model_name)

    def _pp(batch):
        enc = tok(batch["text"], truncation=True, padding="max_length", max_length=max_len)
        enc["labels"] = [label2id[y] for y in batch["label"]]
        return enc

    enc = ds.map(_pp, batched=True, remove_columns=["text","label"])
    cols = ["input_ids","attention_mask","labels"]
    if "token_type_ids" in enc["train"].features: cols.append("token_type_ids")
    enc.set_format(type="torch", columns=cols)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(labels_sorted), id2label=id2label, label2id=label2id
    )

    args = TrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch",
        save_strategy="epoch",                 # simpan manual final
        # load_best_model_at_end=False,
        learning_rate=lr,
        
        save_total_limit=2,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch,
        per_device_eval_batch_size=batch*2,
        load_best_model_at_end=True,
        weight_decay=0.01,
        logging_steps=50,
        report_to="none",
        metric_for_best_model="eval_f1_macro",
        seed=seed,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=enc["train"],
        eval_dataset=enc["validation"],
        tokenizer=tok,
        compute_metrics=compute_metrics_mc,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=20)]
    )

    trainer.train()
    eval_res = trainer.evaluate()

    # Simpan final
    save_dir = os.path.join(out_dir, "final_model")
    os.makedirs(save_dir, exist_ok=True)
    trainer.save_model(save_dir)
    tok.save_pretrained(save_dir)

    # Laporan klasifikasi
    pred_out = trainer.predict(enc["validation"])
    preds = np.argmax(pred_out.predictions, axis=1)
    y_true = pred_out.label_ids
    rep = classification_report(y_true, preds, target_names=[id2label[i] for i in range(len(id2label))], digits=4)
    with open(os.path.join(out_dir, "classification_report.txt"), "w", encoding="utf-8") as f:
        f.write(rep)

    # Simpan peta label
    with open(os.path.join(out_dir, "label_map.json"), "w", encoding="utf-8") as f:
        json.dump({"id2label": id2label, "label2id": {v:k for k,v in id2label.items()}}, f, ensure_ascii=False, indent=2)

    return save_dir, eval_res, id2label, {v:k for k,v in id2label.items()}

# ============ (2) LATIH 4 HEAD ============
paths = {
    "scope":            (f"{DATA_DIR}/train_scope.jsonl",           None),
    "time_granularity": (f"{DATA_DIR}/train_time_granularity.jsonl",None),
    "time_complexity":  (f"{DATA_DIR}/train_time_complexity.jsonl", None),
    "case":             (f"{DATA_DIR}/train_case.jsonl",            None),
}

models = {}
for head, (tr, va) in paths.items():
    print(f"\n--- Training head: {head} ---")
    mdir, ev, i2l, l2i = train_multiclass_from_jsonl(
        BASE_MODEL, tr, va,
        out_dir=os.path.join(OUT_ROOT, head),
        epochs=EPOCHS, lr=LR, batch=BATCH, max_len=MAX_LEN, seed=SEED
    )
    print(f"Eval {head}:", ev)
    models[head] = {"dir": mdir, "eval": ev, "i2l": i2l, "l2i": l2i}

# ============ (3) RUNNER (gabungan 4 head) ============
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification

class AnomalyQuadRunner:
    def __init__(self, scope_dir, gran_dir, cplx_dir, case_dir):
        # Scope
        self.scope_tok = AutoTokenizer.from_pretrained(scope_dir)
        self.scope_cfg = AutoConfig.from_pretrained(scope_dir)
        self.scope_i2l = self.scope_cfg.id2label
        self.scope_m   = AutoModelForSequenceClassification.from_pretrained(scope_dir).eval()

        # Granularity
        self.gran_tok = AutoTokenizer.from_pretrained(gran_dir)
        self.gran_cfg = AutoConfig.from_pretrained(gran_dir)
        self.gran_i2l = self.gran_cfg.id2label
        self.gran_m   = AutoModelForSequenceClassification.from_pretrained(gran_dir).eval()

        # Complexity
        self.cplx_tok = AutoTokenizer.from_pretrained(cplx_dir)
        self.cplx_cfg = AutoConfig.from_pretrained(cplx_dir)
        self.cplx_i2l = self.cplx_cfg.id2label
        self.cplx_m   = AutoModelForSequenceClassification.from_pretrained(cplx_dir).eval()

        # Case (A1..A15 + R1..R6)
        self.case_tok = AutoTokenizer.from_pretrained(case_dir)
        self.case_cfg = AutoConfig.from_pretrained(case_dir)
        self.case_i2l = self.case_cfg.id2label
        self.case_m   = AutoModelForSequenceClassification.from_pretrained(case_dir).eval()

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        for m in [self.scope_m, self.gran_m, self.cplx_m, self.case_m]:
            m.to(self.device)

    @staticmethod
    def _decode(i2l, idx: int):
        if isinstance(i2l, dict):
            return i2l.get(idx) or i2l.get(str(idx)) or f"label_{idx}"
        try:
            return i2l[idx]
        except Exception:
            return f"label_{idx}"

    @torch.no_grad()
    def predict(self, text: str, max_len=128):
        # scope
        s = self.scope_tok(text, truncation=True, padding=True, max_length=max_len, return_tensors="pt").to(self.device)
        s_logits = self.scope_m(**s).logits.squeeze(0).cpu().numpy()
        s_id = int(s_logits.argmax()); s_lab = self._decode(self.scope_i2l, s_id)
        s_conf = float(torch.softmax(torch.from_numpy(s_logits), dim=0)[s_id])

        # gran
        g = self.gran_tok(text, truncation=True, padding=True, max_length=max_len, return_tensors="pt").to(self.device)
        g_logits = self.gran_m(**g).logits.squeeze(0).cpu().numpy()
        g_id = int(g_logits.argmax()); g_lab = self._decode(self.gran_i2l, g_id)
        g_conf = float(torch.softmax(torch.from_numpy(g_logits), dim=0)[g_id])

        # complexity
        x = self.cplx_tok(text, truncation=True, padding=True, max_length=max_len, return_tensors="pt").to(self.device)
        x_logits = self.cplx_m(**x).logits.squeeze(0).cpu().numpy()
        x_id = int(x_logits.argmax()); x_lab = self._decode(self.cplx_i2l, x_id)
        x_conf = float(torch.softmax(torch.from_numpy(x_logits), dim=0)[x_id])

        # case
        c = self.case_tok(text, truncation=True, padding=True, max_length=max_len, return_tensors="pt").to(self.device)
        c_logits = self.case_m(**c).logits.squeeze(0).cpu().numpy()
        c_id = int(c_logits.argmax()); c_lab = self._decode(self.case_i2l, c_id)
        c_conf = float(torch.softmax(torch.from_numpy(c_logits), dim=0)[c_id])

        # route hints sederhana (untuk downstream)
        route = []
        route.append("aggregate_all_machines" if s_lab=="ALL" else "machine_entity_extractor")
        if g_lab in {"HOUR","DAY","WEEK","MONTH","YEAR","SHIFT"}:
            route.append(f"slot_{g_lab.lower()}_extractor")
        if x_lab in {"RANGE_CLEAR","RANGE_REL","WEEK_N","POINT","PERIOD","SHIFT"}:
            route.append(f"time_complexity_{x_lab.lower()}")

        return {
            "scope": {"label": s_lab, "confidence": round(s_conf,4)},
            "time_granularity": {"label": g_lab, "confidence": round(g_conf,4)},
            "time_complexity": {"label": x_lab, "confidence": round(x_conf,4)},
            "case": {"label": c_lab, "confidence": round(c_conf,4)},
            "route_hints": route
        }

# ============ (4) DEMO INFERENCE ============
runner = AnomalyQuadRunner(
    scope_dir=os.path.join(OUT_ROOT, "scope", "final_model"),
    gran_dir =os.path.join(OUT_ROOT, "time_granularity", "final_model"),
    cplx_dir =os.path.join(OUT_ROOT, "time_complexity", "final_model"),
    case_dir =os.path.join(OUT_ROOT, "case", "final_model"),
)

examples = [
    "Apakah ada anomali mesin XP888A dan PQ667 7 hari terakhir?",
    "Tolong tampilkan anomali semua mesin bulan ini",
    "Cek anomali mesin XP888A tanggal 10–20 Januari 2025",
    "Ada anomali semua mesin minggu lalu?",
    "Periksa anomali shift malam untuk XP888A",
]
print("\n=== DEMO PREDIKSI ===")
for q in examples:
    try:
        print("\nQ:", q)
        print(runner.predict(q))
    except Exception as e:
        print("Lewati:", e)

print("\nModel tersimpan di:", OUT_ROOT)


train_scope.jsonl: OK  -> /workspace/anomaly_comprehensive_ds_v2_relclass/train_scope.jsonl
train_time_granularity.jsonl: OK  -> /workspace/anomaly_comprehensive_ds_v2_relclass/train_time_granularity.jsonl
train_time_complexity.jsonl: OK  -> /workspace/anomaly_comprehensive_ds_v2_relclass/train_time_complexity.jsonl
train_case.jsonl: OK  -> /workspace/anomaly_comprehensive_ds_v2_relclass/train_case.jsonl
train_anomaly_comprehensive_relclass_master.jsonl: OK  -> /workspace/anomaly_comprehensive_ds_v2_relclass/train_anomaly_comprehensive_relclass_master.jsonl

--- Training head: scope ---


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1293 [00:00<?, ? examples/s]

Map:   0%|          | 0/177 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/1293 [00:00<?, ? examples/s]

Map:   0%|          | 0/177 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_8273/4062517728.py:113: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Acc,F1 Macro
1,2.293700,1.218014,0.485876,0.414787
2,1.096400,1.069267,0.497175,0.458623
3,1.036500,1.015862,0.514124,0.445210
4,0.944400,0.994830,0.508475,0.461443
5,0.948000,1.002998,0.497175,0.439418
6,0.880500,0.985551,0.502825,0.432510
7,0.887200,0.986304,0.514124,0.470324
8,0.860300,1.009800,0.514124,0.470904
9,0.862400,0.994909,0.519774,0.466301
10,0.843400,1.025597,0.519774,0.510033


Eval case: {'eval_loss': 1.0519472360610962, 'eval_acc': 0.5254237288135594, 'eval_f1_macro': 0.5215579048718484, 'eval_runtime': 0.4126, 'eval_samples_per_second': 429.038, 'eval_steps_per_second': 14.544, 'epoch': 30.0}

=== DEMO PREDIKSI ===

Q: Apakah ada anomali mesin XP888A dan PQ667 7 hari terakhir?
{'scope': {'label': 'NOT_ALL', 'confidence': 0.9999}, 'time_granularity': {'label': 'WEEK', 'confidence': 0.6813}, 'time_complexity': {'label': 'RANGE_REL', 'confidence': 0.9984}, 'case': {'label': 'R2', 'confidence': 0.4796}, 'route_hints': ['machine_entity_extractor', 'slot_week_extractor', 'time_complexity_range_rel']}

Q: Tolong tampilkan anomali semua mesin bulan ini
{'scope': {'label': 'ALL', 'confidence': 0.9999}, 'time_granularity': {'label': 'WEEK', 'confidence': 0.518}, 'time_complexity': {'label': 'PERIOD', 'confidence': 0.7177}, 'case': {'label': 'A11', 'confidence': 0.3945}, 'route_hints': ['aggregate_all_machines', 'slot_week_extractor', 'time_complexity_period']}

Q: C

In [11]:
import re

# --- regex bulan Indonesia & tanggal rentang ---
MONTH_WORDS = r"januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember"
DATE_RANGE_DAY = re.compile(
    rf"\b(\d{{1,2}})\s*(?:-|–|—|s/d|sd|sampai|hingga|to)\s*(\d{{1,2}})\s+({MONTH_WORDS})\s+(20\d{{2}})\b",
    re.I
)

# --- relative phrases (jam/menit/hari/minggu/bulan) ---
REL_PATTERNS = [
    re.compile(r"\b(\d+)\s*(menit|jam|hari|minggu|bulan)\s+terakhir\b", re.I),
    re.compile(r"\b(minggu|bulan|tahun)\s+lalu\b", re.I),
    re.compile(r"\b(MTD|YTD|QTD)\b", re.I),
]

DAYPARTS = {
    "malam": re.compile(r"\bmalam\b", re.I),
    "pagi": re.compile(r"\bpagi\b", re.I),
    "siang": re.compile(r"\bsiang\b", re.I),
    "sore": re.compile(r"\bsore\b", re.I),
    "dini_hari": re.compile(r"\bdini\s*hari\b", re.I),
}

# jam/menit
HOUR_MIN_RANGE = [
    re.compile(r"\b(jam|pukul)\s*(\d{1,2})[:\.]?(\d{2})?\s*(?:-|–|—|s/d|sd|sampai|hingga|to)\s*(\d{1,2})[:\.]?(\d{2})?\b", re.I),
    re.compile(r"\b(\d{1,2})[:\.](\d{2})\s*(?:-|–|—|s/d|sd|sampai|hingga|to)\s*(\d{1,2})[:\.](\d{2})\b", re.I),
]
HOUR_MIN_POINT = [
    re.compile(r"\b(jam|pukul)\s*(\d{1,2})[:\.]?(\d{2})?\b", re.I),
    re.compile(r"\b(\d{1,2})[:\.](\d{2})\b", re.I),
]

# unit hints
DAY_HINT   = re.compile(r"\bhari( ini|an)?\b|\bkemarin\b|\bsenin|selasa|rabu|kamis|jumat|jum’at|sabtu|minggu\b", re.I)
WEEK_HINT  = re.compile(r"\bminggu(an)?\b|\bpekan\b|\bminggu\s+ke-\d+|\bminggu\s+pertama|\bminggu\s+kedua|\bminggu\s+ketiga|\bminggu\s+keempat", re.I)
MONTH_HINT = re.compile(rf"\b(bulan( ini|an| lalu)?|{MONTH_WORDS})\b", re.I)
YEAR_HINT  = re.compile(r"\btahun( ini|an| lalu)?\b|\b20\d{2}\b", re.I)
SHIFT_WORD = re.compile(r"\bshift( pagi| siang| malam)?\b", re.I)

def any_machine_in_text(text):
    # heuristik: ada KAPITAL+digit (XP888A, PQ667, dsb.)
    return bool(re.search(r"\b[A-Z]{2,}[A-Z0-9]*\d+[A-Z0-9]*\b", text))

def normalize_route(route_hints, gran_label, extras):
    """Pastikan hanya slot_* yang sesuai granularity, tambah tag menit/daypart dengan benar."""
    slots = {"slot_hour_extractor","slot_minute_extractor","slot_day_extractor","slot_week_extractor","slot_month_extractor","slot_shift_extractor"}
    keep = []
    # pilih slot sesuai gran
    want = None
    if gran_label=="HOUR":
        want = "slot_minute_extractor" if extras.get("minute_precision") else "slot_hour_extractor"
    elif gran_label=="DAY":   want = "slot_day_extractor"
    elif gran_label=="WEEK":  want = "slot_week_extractor"
    elif gran_label=="MONTH": want = "slot_month_extractor"
    elif gran_label=="SHIFT": want = "slot_shift_extractor"

    if want:
        keep.append(want)

    # non-slot hints pertahankan
    for h in route_hints:
        if h not in slots:
            keep.append(h)

    # minute/daypart tags
    if gran_label=="HOUR":
        if extras.get("hour_range"): keep.append("minute_range=present")
        if extras.get("hour_point"): keep.append("minute_point=present")
    if gran_label=="SHIFT" and extras.get("timeofday"):
        keep.append(f"timeofday={extras['timeofday']}")

    return list(dict.fromkeys(keep))  # unique, keep order

def infer_from_text(text: str):
    t = text.lower()
    extras = {}

    # 1) Rentang tanggal eksplisit: 10–20 Januari 2025
    if DATE_RANGE_DAY.search(t):
        extras["date_range_day"] = True
        return "DAY", "RANGE_CLEAR", extras

    # 2) Range jam/menit
    for pat in HOUR_MIN_RANGE:
        if pat.search(t):
            extras["hour_range"] = True
            extras["minute_precision"] = True
            return "HOUR", "RANGE_CLEAR", extras

    # 3) Relative minutes/hours
    if re.search(r"\b(\d+)\s*menit\s+terakhir\b", t):
        extras["relative_minutes"] = True; extras["minute_precision"] = True
        return "HOUR", "RANGE_REL", extras
    if re.search(r"\b(\d+)\s*jam\s+terakhir\b", t):
        extras["relative_hours"] = True
        return "HOUR", "RANGE_REL", extras

    # 4) Titik jam/menit
    for pat in HOUR_MIN_POINT:
        if pat.search(t):
            extras["hour_point"] = True
            if re.search(r"[:\.]\d{2}", t): extras["minute_precision"] = True
            return "HOUR", "POINT", extras

    # 5) Daypart → SHIFT (termasuk “kemarin malam”)
    for name, pat in DAYPARTS.items():
        if pat.search(t) or (name=="malam" and "kemarin malam" in t):
            extras["timeofday"] = name
            return "SHIFT", "SHIFT", extras

    # 6) Kata "shift"
    if SHIFT_WORD.search(t):
        for name, pat in DAYPARTS.items():
            if pat.search(t): extras["timeofday"] = name; break
        return "SHIFT", "SHIFT", extras

    # 7) Umum DAY/WEEK/MONTH/YEAR
    if WEEK_HINT.search(t):  return "WEEK",  "PERIOD", extras
    if MONTH_HINT.search(t): return "MONTH", "PERIOD", extras
    if DAY_HINT.search(t):   return "DAY",   "PERIOD", extras
    if YEAR_HINT.search(t):  return "YEAR",  "PERIOD", extras
    return None, None, extras

def is_relative_range(text: str):
    return any(p.search(text) for p in REL_PATTERNS)

def map_range_rel_case(scope_label: str, gran_label: str):
    g = (gran_label or "").upper(); s = (scope_label or "").upper()
    if s=="ALL":  return {"DAY":"R4","WEEK":"R5","MONTH":"R6"}.get(g)
    else:         return {"DAY":"R1","WEEK":"R2","MONTH":"R3"}.get(g)

def map_period_case(scope_label: str, gran_label: str, model_case_label: str):
    s = (scope_label or "").upper(); g = (gran_label or "").upper()
    if s=="ALL":  return {"DAY":"A11","WEEK":"A12","MONTH":"A13"}.get(g, model_case_label)
    else:         return {"DAY":"A2","WEEK":"A3","MONTH":"A4"}.get(g, model_case_label)

def map_shift_case(scope_label: str):   return "A15" if (scope_label or "").upper()=="ALL" else "A6"
def map_hour_case(scope_label: str, time_cplx: str, model_case_label: str):
    s = (scope_label or "").upper()
    if time_cplx=="RANGE_CLEAR": return "A14" if s=="ALL" else "A5"
    if time_cplx=="RANGE_REL":   return "R4" if s=="ALL" else "R1"
    return model_case_label

def smart_predict(text: str, runner, conf_floor=0.50):
    raw = runner.predict(text)
    scope = raw["scope"]["label"]
    gran  = raw["time_granularity"]["label"]; gran_conf = raw["time_granularity"]["confidence"]
    cplx  = raw["time_complexity"]["label"]
    case  = raw["case"]["label"]
    route = list(raw.get("route_hints", []))
    reasons = []

    # Scope fix: jika tidak ada entity mesin sama sekali → anggap ALL
    if not any_machine_in_text(text) and scope=="NOT_ALL":
        scope = "ALL"; reasons.append("scope fix: no machine entity -> ALL")

    # Infer dari teks (prioritas: date range/daypart/hour/minute > relative > period)
    g2, x2, extras = infer_from_text(text)
    if g2 and g2 != gran: reasons.append(f"gran {gran}->{g2} (rules)"); gran = g2
    if x2 and x2 != cplx: reasons.append(f"cplx {cplx}->{x2} (rules)"); cplx = x2

    # Jika ada hour/minute/date range eksplisit → **kunci** ke RANGE_CLEAR (jangan diubah oleh frasa relatif)
    if extras.get("hour_range") or extras.get("date_range_day"):
        cplx = "RANGE_CLEAR"; reasons.append("cplx lock: explicit time range -> RANGE_CLEAR")

    # Frasa relatif aktif jika tidak ada lock explicit range
    if is_relative_range(text) and not (extras.get("hour_range") or extras.get("date_range_day")) and cplx!="RANGE_REL":
        cplx = "RANGE_REL"; reasons.append("cplx -> RANGE_REL (relative phrase)")

    # Low-conf gran fallback
    if gran_conf < conf_floor and not g2:
        t = text.lower()
        if WEEK_HINT.search(t)  and gran!="WEEK":  reasons.append(f"gran lowconf {gran_conf}->WEEK");  gran="WEEK"
        elif MONTH_HINT.search(t) and gran!="MONTH": reasons.append(f"gran lowconf {gran_conf}->MONTH"); gran="MONTH"
        elif DAY_HINT.search(t)   and gran!="DAY":   reasons.append(f"gran lowconf {gran_conf}->DAY");   gran="DAY"

    # Case mapping konsisten
    if gran=="SHIFT": case2 = map_shift_case(scope); 
    elif gran=="HOUR": case2 = map_hour_case(scope, cplx, case)
    elif cplx=="RANGE_REL" and gran in {"DAY","WEEK","MONTH"}: case2 = map_range_rel_case(scope, gran)
    elif cplx=="PERIOD"    and gran in {"DAY","WEEK","MONTH"}: case2 = map_period_case(scope, gran, case)
    else: case2 = case
    if case2 and case2 != case: reasons.append(f"case {case}->{case2}"); case = case2

    # Sinkronisasi route_hints
    route = normalize_route(route, gran, extras)

    return {
        "scope": {"label": scope, "confidence": raw["scope"]["confidence"]},
        "time_granularity": {"label": gran, "confidence": raw["time_granularity"]["confidence"]},
        "time_complexity": {"label": cplx, "confidence": raw["time_complexity"]["confidence"]},
        "case": {"label": case, "confidence": raw["case"]["confidence"]},
        "route_hints": route,
        "postprocess_reason": reasons
    }


In [45]:
examples = [
    "Apakah ada anomali mesin XP888A dan PQ667 7 hari terakhir?",
    "Tolong tampilkan anomali semua mesin bulan ini",
    "Cek anomali mesin XP888A tanggal 10–20 Januari 2025",
    "Ada anomali semua mesin minggu lalu?",
    "Periksa anomali shift malam untuk XP888A",
    "apakah ada anomali kemarin malam?",
    "apakah ada anomali di senin minggu lalu pada pukul 8 pagi hingga 10 pagi?",
    "dalam rentang 30 menit terakhir apakah ada keanehan pada mesin?"
]
print("\n=== DEMO PREDIKSI ===")
for q in examples:
    try:
        print("\nQ:", q)
        print(smart_predict(q,runner))
    except Exception as e:
        print("Lewati:", e)

print("\nModel tersimpan di:", OUT_ROOT)


=== DEMO PREDIKSI ===

Q: Apakah ada anomali mesin XP888A dan PQ667 7 hari terakhir?
{'scope': {'label': 'NOT_ALL', 'confidence': 0.9999}, 'time_granularity': {'label': 'DAY', 'confidence': 0.6813}, 'time_complexity': {'label': 'RANGE_REL', 'confidence': 0.9984}, 'case': {'label': 'R1', 'confidence': 0.4796}, 'route_hints': ['slot_day_extractor', 'machine_entity_extractor', 'time_complexity_range_rel'], 'postprocess_reason': ['gran WEEK->DAY (rules)', 'cplx RANGE_REL->PERIOD (rules)', 'cplx -> RANGE_REL (relative phrase)', 'case R2->R1']}

Q: Tolong tampilkan anomali semua mesin bulan ini
{'scope': {'label': 'ALL', 'confidence': 0.9999}, 'time_granularity': {'label': 'MONTH', 'confidence': 0.518}, 'time_complexity': {'label': 'PERIOD', 'confidence': 0.7177}, 'case': {'label': 'A13', 'confidence': 0.3945}, 'route_hints': ['slot_month_extractor', 'aggregate_all_machines', 'time_complexity_period'], 'postprocess_reason': ['gran WEEK->MONTH (rules)', 'case A11->A13']}

Q: Cek anomali mesi

In [13]:
from __future__ import annotations
import re
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from dataclasses import dataclass
import calendar

MONTH_WORDS = "|".join([
    "januari","februari","maret","april","mei","juni",
    "juli","agustus","september","oktober","november","desember"
])
RE_MONTHS_LIST = re.compile(
    rf"\b(?:bulan\s+)?((?:{MONTH_WORDS})(?:\s*(?:,|dan|&)\s*(?:{MONTH_WORDS}))*)\s+(20\d{{2}})\b",
    re.I
)

RE_DAY_RANGE_TWO_MONTHS = re.compile(
    rf"\b(\d{{1,2}})\s*(?:-|–|—|s/d|sd|sampai|hingga|to)\s*(\d{{1,2}})\s+"
    rf"({MONTH_WORDS})\s*(?:dan|&)\s*({MONTH_WORDS})\s+(20\d{{2}})\b",
    re.I
)
def _last_day_of_month(year:int, month:int) -> int:
    return calendar.monthrange(year, month)[1]

ID_MONTHS = {
    "januari":1, "februari":2, "maret":3, "april":4, "mei":5, "juni":6,
    "juli":7, "agustus":8, "september":9, "oktober":10, "november":11, "desember":12
}
DAYPART_RANGES = {"pagi":(5,11),"siang":(11,15),"sore":(15,18),"malam":(18,24),"dini_hari":(0,5)}
DAYNAME_TO_IDX = {"senin":0,"selasa":1,"rabu":2,"kamis":3,"jumat":4,"jum’at":4,"sabtu":5,"hari minggu":6,"minggu hari":6}

RE_DATE_RANGE_DAY = re.compile(
    rf"\b(\d{{1,2}})\s*(?:-|–|—|s/d|sd|sampai|hingga|to)\s*(\d{{1,2}})\s+({'|'.join(ID_MONTHS.keys())})\s+(20\d{{2}})\b",
    re.I
)
RE_HHMM_RANGE = re.compile(r"\b(?:pukul|jam)?\s*(\d{1,2})(?::|\.)(\d{2})\s*(?:-|–|—|s/d|sd|sampai|hingga|to)\s*(\d{1,2})(?::|\.)(\d{2})\b", re.I)
RE_HHMM_RANGE_FLEX = re.compile(
    r"\b(?:pukul|jam)?\s*"
    r"(\d{1,2})(?::|\.)(\d{2})\s*(?:pagi|siang|sore|malam|dini\s*hari)?\s*"
    r"(?:-|–|—|s/d|sd|sampai|hingga|to)\s*"
    r"(\d{1,2})(?::|\.)(\d{2})\s*(?:pagi|siang|sore|malam|dini\s*hari)?\b",
    re.I
)
RE_HH_RANGE   = re.compile(r"\b(?:pukul|jam)\s*(\d{1,2})\s*(?:-|–|—|s/d|sd|sampai|hingga|to)\s*(\d{1,2})\b", re.I)
RE_HH_RANGE_FLEX = re.compile(
    r"\b(?:pukul|jam)\s*"
    r"(\d{1,2})\s*(?:pagi|siang|sore|malam|dini\s*hari)?\s*"
    r"(?:-|–|—|s/d|sd|sampai|hingga|to)\s*"
    r"(\d{1,2})\s*(?:pagi|siang|sore|malam|dini\s*hari)?\b",
    re.I
)
RE_LAST_X     = re.compile(r"\b(\d+)\s*(menit|jam|hari|minggu|bulan)\s+terakhir\b", re.I)
RE_WEEK_LAST  = re.compile(r"\bminggu\s+lalu\b", re.I)
RE_WEEK_THIS  = re.compile(r"\bminggu\s+ini\b", re.I)
RE_MONTH_THIS = re.compile(r"\bbulan\s+ini\b", re.I)
RE_DAYPART    = re.compile(r"\b(pagi|siang|sore|malam|dini\s*hari)\b", re.I)
RE_KEMARIN    = re.compile(r"\bkemarin\b", re.I)
# RE_DAYNAME    = re.compile(r"\b(senin|selasa|rabu|kamis|jumat|jum’at|sabtu|minggu)\b", re.I)
RE_DAYNAME = re.compile(r"\b(senin|selasa|rabu|kamis|jumat|jum’at|sabtu|hari\s+minggu|minggu\s+hari)\b", re.I)
RE_ISO_YMD    = re.compile(r"\b(20\d{2})-(\d{1,2})-(\d{1,2})\b")
RE_DMY_SLASH  = re.compile(r"\b(\d{1,2})/(\d{1,2})/(20\d{2})\b")




# Nama hari TANPA 'minggu' biasa, tetapi IZINKAN 'hari minggu'


# Rentang jam fleksibel: izinkan daypart di sisi kiri/kanan angka


# (opsional) versi HH:MM juga fleksibel terhadap daypart sesudah menit



def _now(): return datetime.now()
def _start_of_week(dt): return (dt - timedelta(days=dt.weekday())).replace(hour=0, minute=0, second=0, microsecond=0)
def _end_of_week(dt): s=_start_of_week(dt); return s + timedelta(days=6, hours=23, minutes=59, seconds=59)

def _daypart_window(base_day: datetime, label: str):
    lb = label.replace(" ", "_").lower()
    h0,h1 = DAYPART_RANGES.get(lb, DAYPART_RANGES["malam"])
    start = base_day.replace(hour=h0, minute=0, second=0, microsecond=0)
    end   = base_day.replace(hour=(h1-1 if h1>0 else 23), minute=59, second=59, microsecond=0)
    return start, end

@dataclass
class TimeWindow:
    start: datetime
    end: datetime
    granularity: str
    kind: str
    note: str = ""

def resolve_time_window(text: str, pred: dict) -> TimeWindow:
    tnow = _now()
    low = text.lower()

    # === [A] Rentang hari eksplisit: 10–20 Januari 2025 ===
    m = RE_DATE_RANGE_DAY.search(low)
    if m:
        d1, d2, mon, year = int(m.group(1)), int(m.group(2)), m.group(3).lower(), int(m.group(4))
        start = datetime(year, ID_MONTHS[mon], d1, 0, 0, 0)
        end   = datetime(year, ID_MONTHS[mon], d2, 23, 59, 59)
        return TimeWindow(start, end, "DAY", "RANGE_CLEAR", "tanggal eksplisit (ID)")

    # === [A2] Rentang hari lintas dua bulan: 1–20 Januari dan Februari 2025 ===
    m = RE_DAY_RANGE_TWO_MONTHS.search(low)
    if m:
        d1, d2 = int(m.group(1)), int(m.group(2))
        mon1, mon2 = m.group(3).lower(), m.group(4).lower()
        year = int(m.group(5))
        m1, m2 = ID_MONTHS[mon1], ID_MONTHS[mon2]
        if (year, m1) > (year, m2):  # pastikan urut
            m1, m2, d1, d2 = m2, m1, d2, d1
        start = datetime(year, m1, d1, 0, 0, 0)
        end   = datetime(year, m2, d2, 23, 59, 59)
        return TimeWindow(start, end, "DAY", "RANGE_CLEAR", f"{d1}–{d2} {mon1}–{mon2} {year}")

    # === [B] Rentang jam eksplisit (default: hari ini) ===
    m = RE_HHMM_RANGE.search(low) or RE_HH_RANGE.search(low)
    if m:
        if len(m.groups()) == 4:
            h1, m1, h2, m2 = map(int, m.groups())
            start = tnow.replace(hour=h1 % 24, minute=m1 % 60, second=0, microsecond=0)
            end   = tnow.replace(hour=h2 % 24, minute=m2 % 60, second=59, microsecond=0)
        else:
            h1, h2 = map(int, m.groups())
            start = tnow.replace(hour=h1 % 24, minute=0, second=0, microsecond=0)
            end   = tnow.replace(hour=h2 % 24, minute=59, second=59, microsecond=0)
        return TimeWindow(start, end, "HOUR", "RANGE_CLEAR", "jam eksplisit (hari ini)")

    # === [C] Frasa relatif: X menit/jam/hari/minggu/bulan terakhir ===
    m = RE_LAST_X.search(low)
    if m:
        qty, unit = int(m.group(1)), m.group(2).lower()
        end = tnow
        if unit.startswith("menit"): start, gran = end - timedelta(minutes=qty), "HOUR"
        elif unit.startswith("jam"):  start, gran = end - timedelta(hours=qty),   "HOUR"
        elif unit.startswith("hari"): start, gran = end - timedelta(days=qty),    "DAY"
        elif unit.startswith("mingg"):start, gran = end - timedelta(weeks=qty),   "WEEK"
        else:                         start, gran = end - relativedelta(months=qty), "MONTH"
        return TimeWindow(start, end, gran, "RANGE_REL", f"{qty} {unit} terakhir")

    # === [D] Kombinasi hari-pekan + daypart/jam (prioritas tinggi) ===
    dn = RE_DAYNAME.search(low)
    is_week_phrase = RE_WEEK_LAST.search(low) or RE_WEEK_THIS.search(low) or "minggu ini" in low
    if dn and is_week_phrase:
        dn_str = dn.group(1).lower()
        ref = (tnow - timedelta(weeks=1)) if RE_WEEK_LAST.search(low) else tnow
        week_start = _start_of_week(ref)
        want_idx = DAYNAME_TO_IDX.get(dn_str, None)
        if want_idx is not None:
            target = week_start + timedelta(days=want_idx)
            start = target.replace(hour=0, minute=0, second=0, microsecond=0)
            end   = target.replace(hour=23, minute=59, second=59, microsecond=0)
            dp = RE_DAYPART.search(low)
            if dp: start, end = _daypart_window(target, dp.group(1))
            m = RE_HHMM_RANGE_FLEX.search(low) or RE_HH_RANGE_FLEX.search(low)
            if m:
                if len(m.groups()) == 4:
                    h1, m1, h2, m2 = map(int, m.groups())
                    start = target.replace(hour=h1 % 24, minute=m1 % 60, second=0, microsecond=0)
                    end   = target.replace(hour=h2 % 24, minute=m2 % 60, second=59, microsecond=0)
                else:
                    h1, h2 = map(int, m.groups())
                    start = target.replace(hour=h1 % 24, minute=0, second=0, microsecond=0)
                    end   = target.replace(hour=h2 % 24, minute=59, second=59, microsecond=0)
            return TimeWindow(start, end, "DAY", "RANGE_CLEAR",
                              f"{dn_str} ({'minggu lalu' if RE_WEEK_LAST.search(low) else 'minggu ini'})")

    # === [E] Daypart (tanpa nama-hari) → hari ini/kemarin ===
    dp = RE_DAYPART.search(low)
    if dp and not (RE_WEEK_LAST.search(low) or RE_WEEK_THIS.search(low) or dn):
        base = (tnow - timedelta(days=1)) if RE_KEMARIN.search(low) else tnow
        base = base.replace(hour=0, minute=0, second=0, microsecond=0)
        start, end = _daypart_window(base, dp.group(1))
        return TimeWindow(start, end, "SHIFT", "SHIFT", f"shift {dp.group(1).lower()}")

    # === [F] Satu tanggal spesifik → sehari penuh ===
    m = RE_ISO_YMD.search(low)
    if m:
        y, mo, d = map(int, m.groups())
        return TimeWindow(datetime(y, mo, d, 0, 0, 0), datetime(y, mo, d, 23, 59, 59),
                          "DAY", "POINT", "tanggal ISO (full day)")
    m = RE_DMY_SLASH.search(low)
    if m:
        d, mo, y = map(int, m.groups())
        return TimeWindow(datetime(y, mo, d, 0, 0, 0), datetime(y, mo, d, 23, 59, 59),
                          "DAY", "POINT", "tanggal dd/mm/yyyy (full day)")

    # === [G] Bulan eksplisit (satu/lebih) + tahun → bulan penuh ===
    m = RE_MONTHS_LIST.search(low)
    if m:
        months_str, year = m.group(1), int(m.group(2))
        # pecah "januari, februari dan maret"
        names = re.split(r"\s*(?:,|dan|&)\s*", months_str, flags=re.I)
        months = sorted(ID_MONTHS[n.lower()] for n in names if n)
        if months:
            m1, m2 = months[0], months[-1]
            start = datetime(year, m1, 1, 0, 0, 0)
            end   = datetime(year, m2, _last_day_of_month(year, m2), 23, 59, 59)
            label = f"{names[0]}–{names[-1]} {year}" if len(months) > 1 else f"{names[0]} {year}"
            return TimeWindow(start, end, "MONTH", "PERIOD", label)

    # === [H] Period generik: minggu/bulan ini ===
    if RE_WEEK_THIS.search(low):
        return TimeWindow(_start_of_week(tnow), tnow, "WEEK", "PERIOD", "minggu ini")
    if RE_WEEK_LAST.search(low):
        ref = tnow - timedelta(weeks=1)
        return TimeWindow(_start_of_week(ref), _end_of_week(ref), "WEEK", "PERIOD", "minggu lalu")
    if RE_MONTH_THIS.search(low):
        start = tnow.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
        return TimeWindow(start, tnow, "MONTH", "PERIOD", "bulan ini")

    # === [I] Fallback ke label model ===
    gran = (pred.get("time_granularity", {}) or {}).get("label", "DAY").upper()
    cplx = (pred.get("time_complexity", {}) or {}).get("label", "PERIOD").upper()
    if gran == "MONTH" and cplx in {"PERIOD", "RANGE_REL"}:
        return TimeWindow(tnow.replace(day=1, hour=0, minute=0, second=0, microsecond=0), tnow,
                          "MONTH", cplx, "fallback model → bulan berjalan")
    if gran == "WEEK" and cplx in {"PERIOD", "RANGE_REL"}:
        return TimeWindow(_start_of_week(tnow), tnow, "WEEK", cplx, "fallback model → minggu berjalan")
    if gran == "DAY" and cplx in {"PERIOD", "RANGE_REL"}:
        return TimeWindow(tnow.replace(hour=0, minute=0, second=0, microsecond=0), tnow,
                          "DAY", cplx, "fallback model → hari ini")

    # === [J] Terakhir: 24 jam terakhir ===
    return TimeWindow(tnow - timedelta(days=1), tnow, "DAY", "RANGE_REL", "fallback 24 jam terakhir")



    # --- (5) Daypart umum (tanpa nama-hari/minggu) → hari ini / kemarin
    dp = RE_DAYPART.search(low)
    if dp and not (RE_WEEK_LAST.search(low) or RE_WEEK_THIS.search(low) or dn):
        base = (tnow - timedelta(days=1)) if RE_KEMARIN.search(low) else tnow
        base = base.replace(hour=0, minute=0, second=0, microsecond=0)
        start,end = _daypart_window(base, dp.group(1))
        return TimeWindow(start,end,"SHIFT","SHIFT",f"shift {dp.group(1).lower()}")

    # --- (6) Satu tanggal spesifik → full day
    m = RE_ISO_YMD.search(low)
    if m:
        y,mo,d = map(int, m.groups())
        start = datetime(y, mo, d, 0,0,0); end = datetime(y, mo, d, 23,59,59)
        return TimeWindow(start,end,"DAY","POINT","tanggal ISO (full day)")
    m = RE_DMY_SLASH.search(low)
    if m:
        d,mo,y = map(int, m.groups())
        start = datetime(y, mo, d, 0,0,0); end = datetime(y, mo, d, 23,59,59)
        return TimeWindow(start,end,"DAY","POINT","tanggal dd/mm/yyyy (full day)")

    # --- (7) Period generik: minggu/bulan ini (fallback ringan)
    if RE_WEEK_THIS.search(low):
        return TimeWindow(_start_of_week(tnow), tnow, "WEEK", "PERIOD", "minggu ini")
    if RE_WEEK_LAST.search(low):
        ref = tnow - timedelta(weeks=1)
        return TimeWindow(_start_of_week(ref), _end_of_week(ref), "WEEK", "PERIOD", "minggu lalu")
    if RE_MONTH_THIS.search(low):
        start = tnow.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
        return TimeWindow(start, tnow, "MONTH", "PERIOD", "bulan ini")

    # --- (8) Fallback ke label model
    gran = (pred.get("time_granularity",{}) or {}).get("label","DAY").upper()
    cplx = (pred.get("time_complexity",{}) or {}).get("label","PERIOD").upper()
    if gran=="MONTH" and cplx in {"PERIOD","RANGE_REL"}:
        return TimeWindow(tnow.replace(day=1,hour=0,minute=0,second=0,microsecond=0), tnow, "MONTH", cplx, "fallback model → bulan berjalan")
    if gran=="WEEK" and cplx in {"PERIOD","RANGE_REL"}:
        return TimeWindow(_start_of_week(tnow), tnow, "WEEK", cplx, "fallback model → minggu berjalan")
    if gran=="DAY" and cplx in {"PERIOD","RANGE_REL"}:
        return TimeWindow(tnow.replace(hour=0,minute=0,second=0,microsecond=0), tnow, "DAY", cplx, "fallback model → hari ini")

    # --- (9) Terakhir: 24 jam terakhir
    return TimeWindow(tnow - timedelta(days=1), tnow, "DAY", "RANGE_REL", "fallback 24 jam terakhir")


In [62]:
examples = [
    "Apakah ada anomali mesin XP888A dan PQ667 7 hari terakhir?",
    "Tolong tampilkan anomali semua mesin bulan ini",
    "Cek anomali mesin XP888A tanggal 10–20 Januari 2025",
    "Ada anomali semua mesin minggu lalu?",
    "Periksa anomali shift malam untuk XP888A",
    "apakah ada anomali kemarin malam?",
    "apakah ada anomali di senin minggu lalu pada pukul 8 pagi hingga 10 pagi?",
    "dalam rentang 30 menit terakhir apakah ada keanehan pada mesin?"
]
print("\n=== DEMO PREDIKSI ===")
for q in examples:
    try:
            # print("\nQ:", q)
            # print(smart_predict(q,runner))
        pred = smart_predict(q, runner)
        tw   = resolve_time_window(q, pred)
        print(q)
        print(pred)
        print()
        print(tw,tw.start, tw.end)
        print()
        
    except Exception as e:
        print("Lewati:", e)

print("\nModel tersimpan di:", OUT_ROOT)

# pred = smart_predict(query_text, runner)
# tw   = resolve_time_window(query_text, pred)
# -> tw.start, tw.end bisa langsung dipakai untuk filter dataframe/database




=== DEMO PREDIKSI ===
Apakah ada anomali mesin XP888A dan PQ667 7 hari terakhir?
{'scope': {'label': 'NOT_ALL', 'confidence': 0.9999}, 'time_granularity': {'label': 'DAY', 'confidence': 0.6813}, 'time_complexity': {'label': 'RANGE_REL', 'confidence': 0.9984}, 'case': {'label': 'R1', 'confidence': 0.4796}, 'route_hints': ['slot_day_extractor', 'machine_entity_extractor', 'time_complexity_range_rel'], 'postprocess_reason': ['gran WEEK->DAY (rules)', 'cplx RANGE_REL->PERIOD (rules)', 'cplx -> RANGE_REL (relative phrase)', 'case R2->R1']}

TimeWindow(start=datetime.datetime(2025, 10, 16, 9, 11, 43, 502249), end=datetime.datetime(2025, 10, 23, 9, 11, 43, 502249), granularity='DAY', kind='RANGE_REL', note='7 hari terakhir') 2025-10-16 09:11:43.502249 2025-10-23 09:11:43.502249

Tolong tampilkan anomali semua mesin bulan ini
{'scope': {'label': 'ALL', 'confidence': 0.9999}, 'time_granularity': {'label': 'MONTH', 'confidence': 0.518}, 'time_complexity': {'label': 'PERIOD', 'confidence': 0.717

In [57]:
# ============================================================
# Anomaly Query Engine: dari teks → query → tabel anomali
# ============================================================
from __future__ import annotations
import re
from dataclasses import dataclass
from typing import List, Dict, Any, Optional
from datetime import datetime
import pandas as pd
import numpy as np

# ----------------------------
# 0) UTIL — entity mesin & dt
# ----------------------------
RE_MACHINE = re.compile(r"\b[A-Z]{2,}[A-Z0-9]*\d+[A-Z0-9]*\b")

def extract_machines_from_text(text: str) -> List[str]:
    """Ekstrak ID mesin seperti XP888A, PQ667, dsb."""
    return list(dict.fromkeys(RE_MACHINE.findall(text.upper())))

def ensure_datetime(series) -> pd.Series:
    """Baca kolom waktu yang bisa berupa string ISO, dd/mm/yyyy HH:MM, atau Excel-like."""
    # ganti koma → titik kalau ada (untuk angka yang tak sengaja masuk)
    return pd.to_datetime(series, errors="coerce", dayfirst=True)

# ---------------------------------------------------------
# 1) LOADER (opsional) — ubah ke sumber data kamu sendiri
# ---------------------------------------------------------
def load_thresholds_from_excel(path: str, sheet: Optional[str] = None) -> pd.DataFrame:
    """
    Membaca tabel batas min–max.
    Harus menghasilkan kolom: machine_id, min_value, max_value
    """
    raw = pd.read_excel(path, sheet_name=sheet)
    # DETEKSI otomatis nama kolom yang mirip
    col_map = {}
    for c in raw.columns:
        lc = str(c).strip().lower()
        if "tag" in lc or "machine" in lc: col_map[c] = "machine_id"
        elif "min" in lc:                  col_map[c] = "min_value"
        elif "max" in lc:                  col_map[c] = "max_value"
    df = raw.rename(columns=col_map)[["machine_id","min_value","max_value"]].copy()
    df["machine_id"] = df["machine_id"].astype(str).str.upper().str.strip()
    return df

def load_timeseries_from_csv(path: str) -> pd.DataFrame:
    """
    Contoh loader timeseries. Harus hasilkan kolom: machine_id, ts, value
    """
    df = pd.read_csv(path)
    # normalisasi nama kolom
    col_map = {}
    for c in df.columns:
        lc = str(c).strip().lower().replace(" ", "")
        if lc in {"machine_id","machineid","tag","tagnumber"}: col_map[c] = "machine_id"
        elif lc in {"ts","timestamp","datetime","timestamp/datetime"}: col_map[c] = "ts"
        elif "value" in lc: col_map[c] = "value"
    df = df.rename(columns=col_map)[["machine_id","ts","value"]].copy()

    # normalisasi nilai
    df["machine_id"] = df["machine_id"].astype(str).str.upper().str.strip()
    # ganti koma desimal → titik bila perlu
    if df["value"].dtype == object:
        df["value"] = (df["value"].astype(str)
                                 .str.replace(",", ".", regex=False)
                                 .str.replace(" ", "", regex=False))
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df["ts"] = ensure_datetime(df["ts"])
    df = df.dropna(subset=["machine_id","ts","value"])
    return df

# -------------------------------------------------------------------
# 2) CORE — filter data, join thresholds, dan tandai baris anomali
# -------------------------------------------------------------------
@dataclass
class AnomalyResult:
    query_text: str
    start: datetime
    end: datetime
    scope: str
    machines: List[str]
    summary: pd.DataFrame
    details: pd.DataFrame
    
def debug_ts_coverage(df_ts):
    print()
    # print("Dtypes:", df_ts.dtypes.to_dict())
    # print("ts.min:", df_ts["ts"].min(), "ts.max:", df_ts["ts"].max())
    # vc = df_ts["ts"].dt.date.value_counts().sort_index()
    # print("Jumlah baris per tanggal (top 40):")
    # print(vc.head(40).to_string())

def detect_anomalies_for_window(
    query_text: str,
    pred: Dict[str, Any],
    tw,                              # TimeWindow dari resolve_time_window
    df_ts: pd.DataFrame,             # kolom: machine_id, ts, value
    df_thr: pd.DataFrame,            # kolom: machine_id, min_value, max_value
    machines_override: Optional[List[str]] = None,
) -> AnomalyResult:
    """
    Jalankan deteksi anomali berdasarkan rentang waktu & mesin.
    """
    scope = (pred.get("scope") or {}).get("label", "ALL")

    # Mesin dari teks; jika scope ALL, pakai semua dari thresholds
    if machines_override is not None:
        machines = [m.upper() for m in machines_override]
    else:
        found = extract_machines_from_text(query_text)
        if scope == "ALL" or not found:
            machines = sorted(df_thr["machine_id"].unique().tolist())
        else:
            # batasi hanya yang ada di thresholds
            machines = [m for m in found if m in set(df_thr["machine_id"])]

    if not machines:
        # jika tidak ada sama sekali di thresholds, pakai semua yg ada di data
        machines = sorted(df_ts["machine_id"].unique().tolist())
    # debug_ts_coverage(df_ts)
    # Filter waktu & mesin
    mask_time = (df_ts["ts"] >= tw.start) & (df_ts["ts"] <= tw.end)
    mask_mach = df_ts["machine_id"].isin(machines)
    df_win = df_ts.loc[mask_time & mask_mach].copy()

    # Join thresholds
    thr = df_thr.copy()
    thr["machine_id"] = thr["machine_id"].astype(str).str.upper()
    df_win = df_win.merge(thr, on="machine_id", how="left")

    # Tandai anomali
    df_win["is_low"]  = df_win["value"] < df_win["min_value"]
    df_win["is_high"] = df_win["value"] > df_win["max_value"]
    df_win["is_anom"] = df_win["is_low"] | df_win["is_high"]
    # 1) Apakah semua ts jatuh di hari ke-1?
    days = df_ts["ts"].dt.day.value_counts().sort_index()
    # print("Distribusi hari dalam sebulan:\n", days.to_string())
    
    # 2) Apakah ada jejak operasi period/bulanan?
    # (Cari string mencurigakan di file/nb code kamu)
    suspicious = [
        "to_period('M')", "to_period(\"M\")",
        "astype('datetime64[M]')", 'astype("datetime64[M]")',
        "floor('MS')", 'floor("MS")',
        "floor('M')", 'floor("M")',
        "Grouper(", "freq='M'", 'freq="M"', "freq='MS'", 'freq="MS"'
    ]
    # manual search di editor kamu; ini hanya daftar yang perlu dicari
    print("Cari di kode: ", suspicious)

    # Siapkan ringkasan
    # - hitung total baris, total anomali, first/last ts per mesin
    agg = (df_win
           .groupby("machine_id")
           .agg(total_points=("value","size"),
                anomalies=("is_anom","sum"),
                first_ts=("ts","min"),
                last_ts=("ts","max"))
           .reset_index())
    # Tambahkan flag “ada anomali?”
    agg["has_anomaly"] = agg["anomalies"] > 0

    # Detil anomali saja
    details = (df_win.loc[df_win["is_anom"],
                          ["machine_id","ts","value","min_value","max_value","is_low","is_high"]]
                      .sort_values(["machine_id","ts"]))
    
    return AnomalyResult(
        query_text=query_text,
        start=tw.start,
        end=tw.end,
        scope=scope,
        machines=machines,
        summary=agg.sort_values(["has_anomaly","machine_id"], ascending=[False,True]),
        details=details
    )

# -------------------------------------------------------------------
# 3) WRAPPER — dari teks langsung ke hasil
# -------------------------------------------------------------------
def run_anomaly_query(
    text: str,
    runner,                         # AnomalyQuadRunner milikmu
    df_ts: pd.DataFrame,
    df_thr: pd.DataFrame,
) -> AnomalyResult:
    """
    1) smart_predict → 2) resolve_time_window → 3) detect_anomalies_for_window
    """
    # print("range:", df_ts["ts"].min(), "→", df_ts["ts"].max())
    # print(df_ts["ts"].dt.date.value_counts().sort_index().head(40))
    # print("unique days:", df_ts["ts"].dt.date.nunique())
    # print(df_ts["ts"].dt.day.value_counts().sort_index())  # kalau nyangkut di '1', ketahuan

    pred = smart_predict(text, runner)
    tw   = resolve_time_window(text, pred)
    res  = detect_anomalies_for_window(text, pred, tw, df_ts, df_thr)
    return res

# -------------------------------------------------------------------
# 4) CONTOH PEMAKAIAN
# -------------------------------------------------------------------
# (a) Kalau datamu sdh ada sebagai DataFrame, lompat ke (c)
# df_thr = load_thresholds_from_excel("thresholds.xlsx", sheet="Sheet1")
# df_ts  = load_timeseries_from_csv("timeseries.csv")

# (b) Pastikan kolom ts bertipe datetime dan machine_id uppercase
# df_ts["ts"] = ensure_datetime(df_ts["ts"])
# df_ts["machine_id"] = df_ts["machine_id"].str.upper().str.strip()
# df_thr["machine_id"] = df_thr["machine_id"].str.upper().str.strip()

# (c) Jalankan query
# q = "Apakah ada anomali mesin XP888A dan PQ667 7 hari terakhir?"
# result = run_anomaly_query(q, runner, df_ts, df_thr)

# (d) Akses hasil
# print("WINDOW :", result.start, "→", result.end)
# print("MESIN   :", result.machines)
# print("\nRINGKASAN:")
# print(result.summary)
# print("\nDETIL ANOMALI:")
# print(result.details)


In [23]:
import pandas as pd
import numpy as np
from pathlib import Path

# ======================================
# CONFIG
# ======================================
EXCEL_PATH = "/workspace/wefesfes2.xlsx"   # update kalau filenya di path lain
OUT_CSV    = "/workspace/df_raw2.csv"      # hasil bersih

# ======================================
# HELPER: normalisasi nama kolom
# ======================================
def norm(s: str) -> str:
    return (
        str(s)
        .strip()
        .lower()
        .replace(" ", "")
        .replace("_", "")
        .replace("-", "")
        .replace("/", "")
        .replace(":", "")
    )

# kandidat nama kolom untuk mapping fleksibel
CAND_MACHINE   = {"tagnumber", "tagnumbe", "tag", "machine", "mesin", "unit", "machineid", "tagid"}
CAND_TIMESTAMP = {
    "timestampdatetime", "timestampdatetime", "timestamp", "datetime", "waktu",
    "timestamptime", "timestamptimestamp", "timestampdatetime", "timestampdatetime",
    "timestampdatetime", "timestamptime", "datetime", "datetimestamp", "timestampdatetime"
}
# kita broad-kan sedikit biar aman
CAND_TIMESTAMP = CAND_TIMESTAMP.union({"timestampdatetime","timestampdatetime","timestampdatetime","timestampdatetime","timestampdatetime","timestampdatetime","timestamps","time","date","datetimestamp","timestampdatetime","timestampdatetim"}) 
CAND_VALUE     = {"value", "nilai", "val", "reading", "pv", "processvalue"}

def map_columns(df_sheet: pd.DataFrame):
    """
    Cari kolom machine_id / ts / value di satu sheet.
    Return dict {'machine_id': colA, 'ts': colB, 'value': colC}
    atau {} kalau ga ketemu.
    """
    cols_norm = {c: norm(c) for c in df_sheet.columns}
    # kebalikannya buat lookup cepat
    inv = {}
    for orig, nrm in cols_norm.items():
        inv.setdefault(nrm, orig)

    col_map = {}

    # machine_id
    for nrm, orig in inv.items():
        if nrm in CAND_MACHINE:
            col_map["machine_id"] = orig
            break

    # timestamp
    for nrm, orig in inv.items():
        if nrm in CAND_TIMESTAMP:
            col_map["ts"] = orig
            break

    # value
    for nrm, orig in inv.items():
        if nrm in CAND_VALUE:
            col_map["value"] = orig
            break

    needed = {"machine_id", "ts", "value"}
    if not needed.issubset(col_map.keys()):
        return {}

    return col_map


def to_float_any(x):
    """
    Ubah string '123,45' -> 123.45 (float)
    Kalau sudah float/int biarin saja.
    """
    if isinstance(x, str):
        x = x.replace(",", ".")
    return pd.to_numeric(x, errors="coerce")


def clean_one_sheet(df_sheet: pd.DataFrame, sheet_name: str):
    """
    - deteksi kolom penting
    - rename -> ['machine_id','ts','value']
    - parse waktu, normalisasi angka
    - buang null
    - return df bersih (index reset)
    """
    cmap = map_columns(df_sheet)
    if not cmap:
        print(f"[SKIP] Sheet '{sheet_name}' tidak punya kolom wajib (machine_id / ts / value). Kolom yg ada: {list(df_sheet.columns)}")
        return pd.DataFrame(columns=["machine_id","ts","value"])

    sub = df_sheet[[cmap["machine_id"], cmap["ts"], cmap["value"]]].copy()
    sub.columns = ["machine_id", "ts", "value"]

    # parse waktu
    sub["ts"] = pd.to_datetime(sub["ts"], errors="coerce", dayfirst=True)

    # normalisasi nilai angka
    sub["value"] = sub["value"].apply(to_float_any)

    # bersihkan
    sub["machine_id"] = sub["machine_id"].astype(str).str.upper().str.strip()
    sub = sub.dropna(subset=["ts","value","machine_id"]).copy()

    # sort & dedup per mesin+timestamp
    sub = (sub
           .sort_values(["machine_id","ts"])
           .drop_duplicates(subset=["machine_id","ts"], keep="last")
           .reset_index(drop=True))

    sub["source_sheet"] = sheet_name  # supaya kita tahu asalnya
    return sub


# ======================================
# 1. LOAD SEMUA SHEET
# ======================================
xls = pd.ExcelFile(EXCEL_PATH)
all_clean = []

print("Sheets terdeteksi:", xls.sheet_names)
for sh in xls.sheet_names:
    raw_sheet = pd.read_excel(xls, sheet_name=sh)
    print(f"\n=== Processing sheet: {sh} ===")
    print("Kolom asli:", list(raw_sheet.columns))

    cleaned = clean_one_sheet(raw_sheet, sh)

    print(f" -> Rows setelah clean: {len(cleaned)}")
    if len(cleaned):
        print(cleaned.head(3))
    all_clean.append(cleaned)

# gabung semua sheet
df_raw = pd.concat(all_clean, ignore_index=True)

# fallback kalau kosong total
if df_raw.empty:
    raise RuntimeError("Tidak ada data valid setelah proses semua sheet. Cek nama kolom di Excel.")

# pastikan tipe
df_raw["ts"] = pd.to_datetime(df_raw["ts"], errors="coerce")
df_raw = df_raw.dropna(subset=["ts","value","machine_id"])

# resort global dan dedup global
df_raw = (
    df_raw
    .sort_values(["machine_id","ts"])
    .drop_duplicates(subset=["machine_id","ts"], keep="last")
    .reset_index(drop=True)
)

print("\n=== DF_RAW FINAL (HEAD) ===")
print(df_raw.head(10))
print("shape:", df_raw.shape)

# ======================================
# 2. DIAGNOSTIK GLOBAL
# ======================================
print("\n=== DIAGNOSTIK GLOBAL ===")
machines_unique = df_raw["machine_id"].unique().tolist()
print("Mesin unik:", machines_unique)

ts_min = df_raw["ts"].min()
ts_max = df_raw["ts"].max()
print("Rentang waktu global : ", ts_min, "→", ts_max)

# berapa hari unik?
unique_days = df_raw["ts"].dt.date.nunique()
print("Jumlah hari unik      :", unique_days)

# group per tanggal & mesin → hitung point/hari per mesin
per_day = (
    df_raw
    .groupby([df_raw["ts"].dt.date, "machine_id"])
    .size()
    .rename("count_per_day")
    .reset_index()
)

print("\nContoh distribusi jumlah titik per (hari, mesin):")
print(per_day.head(20))

# estimasi rata-rata sample per hari per mesin
daily_stats = per_day.groupby("machine_id")["count_per_day"].describe()
print("\nStatistik jumlah titik /hari /mesin:")
print(daily_stats)

# coverage harian (berapa row per hari calendar, ignore mesin)
by_date = df_raw.groupby(df_raw["ts"].dt.date).size()
print("\nJumlah row per hari (first 60 hari):")
print(by_date.head(60))

# ======================================
# 3. CEK INTERVAL (APAKAH ~5 MENIT?)
# ======================================
print("\n=== CEK INTERVAL SAMPLING PER MESIN ===")
def check_freq(g):
    g = g.sort_values("ts")
    diffs = g["ts"].diff().dropna()
    # ambil delta yg paling umum (mode approx)
    top = (
        diffs.value_counts()
        .head(5)
    )
    return top

freq_report = {}
for m_id, g in df_raw.groupby("machine_id"):
    freq_report[m_id] = check_freq(g)

for m_id, vc in freq_report.items():
    print(f"\nMesin {m_id} diff(ts) paling sering muncul:")
    print(vc)

# kalau ingin lihat apakah rata-rata diff ~5 menit:
for m_id, g in df_raw.groupby("machine_id"):
    g = g.sort_values("ts")
    diffs_min = g["ts"].diff().dropna().dt.total_seconds() / 60.0
    if len(diffs_min):
        print(f"\nMesin {m_id} avg step (menit): {diffs_min.mean():.2f} (std {diffs_min.std():.2f}) n={len(diffs_min)})")
    else:
        print(f"\nMesin {m_id} hanya punya 1 titik waktu (nggak bisa hitung interval).")

# ======================================
# 4. SIMPAN KE CSV / DIPAKAI OLEH ENGINE
# ======================================
Path(OUT_CSV).parent.mkdir(parents=True, exist_ok=True)
df_raw.to_csv(OUT_CSV, index=False)

print(f"\n=== DONE ===")
print(f"Hasil data bersih disimpan di: {OUT_CSV}")

# kalau mau langsung pakai ke engine anomaly:
# df_ts  = df_raw[["machine_id","ts","value"]].copy()
# df_thr = <data threshold kamu>  # misal DataFrame manual min/max tiap mesin


Sheets terdeteksi: ['Sheet1']

=== Processing sheet: Sheet1 ===
Kolom asli: ['Tag Number', 'Timestamp/Date Time', 'Value']
 -> Rows setelah clean: 6150
  machine_id                  ts     value source_sheet
0      PQ667 2025-01-01 11:42:57  127.5938       Sheet1
1      PQ667 2025-01-01 11:45:20  127.5957       Sheet1
2      PQ667 2025-01-01 12:06:20  127.7494       Sheet1

=== DF_RAW FINAL (HEAD) ===
  machine_id                  ts     value source_sheet
0      PQ667 2025-01-01 11:42:57  127.5938       Sheet1
1      PQ667 2025-01-01 11:45:20  127.5957       Sheet1
2      PQ667 2025-01-01 12:06:20  127.7494       Sheet1
3      PQ667 2025-01-01 12:18:20  144.1829       Sheet1
4      PQ667 2025-01-01 12:25:20  127.6525       Sheet1
5      PQ667 2025-01-01 12:52:20  127.6584       Sheet1
6      PQ667 2025-01-01 13:19:20  127.8219       Sheet1
7      PQ667 2025-01-01 13:22:20  127.7560       Sheet1
8      PQ667 2025-01-01 13:53:20  127.9142       Sheet1
9      PQ667 2025-01-01 14:12:20  1

In [52]:
import pandas as pd
from datetime import datetime

# =========================
# util bantu
# =========================
def ensure_datetime(series) -> pd.Series:
    """
    Konversi kolom waktu jadi datetime64[ns].
    Mencoba format umum: ISO, dd/mm/yyyy HH:MM, dsb.
    dayfirst=True supaya '01/02/2025' dibaca 1 Feb.
    """
    return pd.to_datetime(series, errors="coerce", dayfirst=True)


# =========================
# loader spesifik df_raw2.csv
# =========================
import pandas as pd

def load_timeseries_from_df_raw2(csv_path: str) -> pd.DataFrame:
    """
    Baca df_raw2.csv jadi dataframe clean:
    - machine_id (string uppercase)
    - ts (datetime benar)
    - value (float)
    """
    # 1. Baca csv
    df = pd.read_csv(csv_path)

    # 2. Map kolom (fallback kalau nama beda-beda)
    def _norm_colname(name: str) -> str:
        return (
            str(name)
            .strip()
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
            .replace("/", "")
        )

    col_map = {}
    for c in df.columns:
        norm_c = _norm_colname(c)

        if norm_c in {"machineid","machine","mesin","unit","tag","tagnumber","tagnumbe"}:
            col_map[c] = "machine_id"
        elif norm_c in {
            "ts","timestamp","datetime","timestamptime",
            "timestampdatetime","datetimestamp"
        }:
            col_map[c] = "ts"
        elif (norm_c in {"value","nilai","val","pv","reading","processvalue"}) or \
             ("value" in norm_c):
            col_map[c] = "value"
        elif c in {"machine_id", "ts", "value"}:
            col_map[c] = c

    df = df.rename(columns=col_map)

    required_cols = {"machine_id","ts","value"}
    if not required_cols.issubset(df.columns):
        raise ValueError(
            f"Kolom wajib tidak lengkap. Kolom ada: {list(df.columns)}; "
            f"harus ada: {required_cols}"
        )

    df = df[["machine_id","ts","value"]].copy()

    # machine_id → uppercase rapi
    df["machine_id"] = (
        df["machine_id"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    # value → float aman (support koma)
    if df["value"].dtype == object:
        df["value"] = (
            df["value"]
            .astype(str)
            .str.replace(",", ".", regex=False)
            .str.replace(" ", "", regex=False)
        )
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    # === PARSING TIMESTAMP YANG BENAR ===
    # Kita coba asumsikan formatnya MM/DD/YYYY HH:MM[:SS]
    # Jangan dayfirst!
    df["ts"] = pd.to_datetime(
        df["ts"],
        errors="coerce",
        dayfirst=False,               # <-- ini perubahan penting
        infer_datetime_format=True    # biar cepat & akurat
    )

    # 4. Cleanup dasar
    df = (
        df.dropna(subset=["machine_id","ts","value"])
          .sort_values(["machine_id","ts"])
          .drop_duplicates(subset=["machine_id","ts"], keep="last")
          .reset_index(drop=True)
    )

    # 5. Debug coverage untuk ngecek bener ga sekarang
    debug_ts_coverage(df)

    return df


def debug_ts_coverage(df_ts, max_print=50):
    # print("=== DEBUG TS COVERAGE ===")
    # print("dtypes :", df_ts.dtypes.to_dict())
    # print("ts.min:", df_ts["ts"].min(), "ts.max:", df_ts["ts"].max())
    # print()

    # vc_date = (
    #     df_ts["ts"]
    #     .dt.date
    #     .value_counts()
    #     .sort_index()
    # )
    # print("Jumlah baris per tanggal:")
    # print(vc_date.head(max_print).to_string())
    # print("unique days:", vc_date.index.nunique())
    # print()

    # vc_month = (
    #     df_ts["ts"]
    #     .dt.to_period("M")
    #     .value_counts()
    #     .sort_index()
    # )
    # print("Jumlah baris per bulan:")
    # print(vc_month.to_string())

    # print("\nDistribusi nomor-hari (1..31) dalam df_ts:")
    # vc_daynum = df_ts["ts"].dt.day.value_counts().sort_index()
    # print(vc_daynum.to_string())
    # print("=========================\n")



# =========================
# contoh pakai
# =========================
# path CSV kamu (replace kalau beda lokasi)
import importlib
import pandas as pd
importlib.reload(pd) 
df_ts = load_timeseries_from_df_raw2("/workspace/df_raw2.csv")

print("df_ts.head():")
print(df_ts.head())

print("\nunique machine_id:", df_ts["machine_id"].unique())
print("total rows:", len(df_ts))


IndentationError: expected an indented block after function definition on line 113 (2082566773.py, line 150)

In [58]:
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch
import os
examples = [
    "Apakah ada anomali mesin XP888A dan PQ667 7 hari terakhir?",
    "Tolong tampilkan anomali semua mesin bulan ini",
    "Cek anomali mesin XP888A tanggal 1–20 Januari 2025",
    "Cek anomali mesin XP888A tanggal 1–20 Januari dan februari 2025",
    "Cek anomali mesin  bulan Januari dan februari 2025",
    "Cek anomali mesin  bulan Januari 2025",
    "Cek anomali mesin  bulan februari 2025",
    "Ada anomali semua mesin minggu lalu?",
    "Periksa anomali shift malam untuk XP888A",
    "apakah ada anomali kemarin malam?",
    "apakah ada anomali di senin minggu lalu pada pukul 8 pagi hingga 10 pagi?",
    "dalam rentang 30 menit terakhir apakah ada keanehan pada mesin?"
]
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification

class AnomalyQuadRunner:
    def __init__(self, scope_dir, gran_dir, cplx_dir, case_dir):
        # Scope
        self.scope_tok = AutoTokenizer.from_pretrained(scope_dir)
        self.scope_cfg = AutoConfig.from_pretrained(scope_dir)
        self.scope_i2l = self.scope_cfg.id2label
        self.scope_m   = AutoModelForSequenceClassification.from_pretrained(scope_dir).eval()

        # Granularity
        self.gran_tok = AutoTokenizer.from_pretrained(gran_dir)
        self.gran_cfg = AutoConfig.from_pretrained(gran_dir)
        self.gran_i2l = self.gran_cfg.id2label
        self.gran_m   = AutoModelForSequenceClassification.from_pretrained(gran_dir).eval()

        # Complexity
        self.cplx_tok = AutoTokenizer.from_pretrained(cplx_dir)
        self.cplx_cfg = AutoConfig.from_pretrained(cplx_dir)
        self.cplx_i2l = self.cplx_cfg.id2label
        self.cplx_m   = AutoModelForSequenceClassification.from_pretrained(cplx_dir).eval()

        # Case (A1..A15 + R1..R6)
        self.case_tok = AutoTokenizer.from_pretrained(case_dir)
        self.case_cfg = AutoConfig.from_pretrained(case_dir)
        self.case_i2l = self.case_cfg.id2label
        self.case_m   = AutoModelForSequenceClassification.from_pretrained(case_dir).eval()

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        for m in [self.scope_m, self.gran_m, self.cplx_m, self.case_m]:
            m.to(self.device)

    @staticmethod
    def _decode(i2l, idx: int):
        if isinstance(i2l, dict):
            return i2l.get(idx) or i2l.get(str(idx)) or f"label_{idx}"
        try:
            return i2l[idx]
        except Exception:
            return f"label_{idx}"

    @torch.no_grad()
    def predict(self, text: str, max_len=128):
        # scope
        s = self.scope_tok(text, truncation=True, padding=True, max_length=max_len, return_tensors="pt").to(self.device)
        s_logits = self.scope_m(**s).logits.squeeze(0).cpu().numpy()
        s_id = int(s_logits.argmax()); s_lab = self._decode(self.scope_i2l, s_id)
        s_conf = float(torch.softmax(torch.from_numpy(s_logits), dim=0)[s_id])

        # gran
        g = self.gran_tok(text, truncation=True, padding=True, max_length=max_len, return_tensors="pt").to(self.device)
        g_logits = self.gran_m(**g).logits.squeeze(0).cpu().numpy()
        g_id = int(g_logits.argmax()); g_lab = self._decode(self.gran_i2l, g_id)
        g_conf = float(torch.softmax(torch.from_numpy(g_logits), dim=0)[g_id])

        # complexity
        x = self.cplx_tok(text, truncation=True, padding=True, max_length=max_len, return_tensors="pt").to(self.device)
        x_logits = self.cplx_m(**x).logits.squeeze(0).cpu().numpy()
        x_id = int(x_logits.argmax()); x_lab = self._decode(self.cplx_i2l, x_id)
        x_conf = float(torch.softmax(torch.from_numpy(x_logits), dim=0)[x_id])

        # case
        c = self.case_tok(text, truncation=True, padding=True, max_length=max_len, return_tensors="pt").to(self.device)
        c_logits = self.case_m(**c).logits.squeeze(0).cpu().numpy()
        c_id = int(c_logits.argmax()); c_lab = self._decode(self.case_i2l, c_id)
        c_conf = float(torch.softmax(torch.from_numpy(c_logits), dim=0)[c_id])

        # route hints sederhana (untuk downstream)
        route = []
        route.append("aggregate_all_machines" if s_lab=="ALL" else "machine_entity_extractor")
        if g_lab in {"HOUR","DAY","WEEK","MONTH","YEAR","SHIFT"}:
            route.append(f"slot_{g_lab.lower()}_extractor")
        if x_lab in {"RANGE_CLEAR","RANGE_REL","WEEK_N","POINT","PERIOD","SHIFT"}:
            route.append(f"time_complexity_{x_lab.lower()}")

        return {
            "scope": {"label": s_lab, "confidence": round(s_conf,4)},
            "time_granularity": {"label": g_lab, "confidence": round(g_conf,4)},
            "time_complexity": {"label": x_lab, "confidence": round(x_conf,4)},
            "case": {"label": c_lab, "confidence": round(c_conf,4)},
            "route_hints": route
        }

runner = AnomalyQuadRunner(
    scope_dir=os.path.join(OUT_ROOT, "scope", "final_model"),
    gran_dir =os.path.join(OUT_ROOT, "time_granularity", "final_model"),
    cplx_dir =os.path.join(OUT_ROOT, "time_complexity", "final_model"),
    case_dir =os.path.join(OUT_ROOT, "case", "final_model"),
)
print("\n=== DEMO PREDIKSI ===")
df_thr = pd.DataFrame({
    "machine_id": ["XP888A","PQ667"],
    "max_value": [10, 140],
    "min_value": [0, 110],
})
df_ts  = load_timeseries_from_df_raw2("/workspace/df_raw2.csv")
for q in examples:
    try:
            # print("\nQ:", q)
            # print(smart_predict(q,runner))
        print(q)
        print(run_anomaly_query(q,runner,df_ts=df_ts,df_thr=df_thr))
        print()
        
    except Exception as e:
        print("Lewati:", e)
OUT_ROOT   = "./runs_anomaly_quad_relclass"                    # folder output
print("\nModel tersimpan di:", OUT_ROOT)

# pred = smart_predict(query_text, runner)
# tw   = resolve_time_window(query_text, pred)
# -> tw.start, tw.end bisa langsung dipakai untuk filter dataframe/database




=== DEMO PREDIKSI ===

Apakah ada anomali mesin XP888A dan PQ667 7 hari terakhir?
Cari di kode:  ["to_period('M')", 'to_period("M")', "astype('datetime64[M]')", 'astype("datetime64[M]")', "floor('MS')", 'floor("MS")', "floor('M')", 'floor("M")', 'Grouper(', "freq='M'", 'freq="M"', "freq='MS'", 'freq="MS"']
AnomalyResult(query_text='Apakah ada anomali mesin XP888A dan PQ667 7 hari terakhir?', start=datetime.datetime(2025, 10, 17, 6, 55, 10, 279802), end=datetime.datetime(2025, 10, 24, 6, 55, 10, 279802), scope='NOT_ALL', machines=['XP888A', 'PQ667'], summary=Empty DataFrame
Columns: [machine_id, total_points, anomalies, first_ts, last_ts, has_anomaly]
Index: [], details=Empty DataFrame
Columns: [machine_id, ts, value, min_value, max_value, is_low, is_high]
Index: [])

Tolong tampilkan anomali semua mesin bulan ini
Cari di kode:  ["to_period('M')", 'to_period("M")', "astype('datetime64[M]')", 'astype("datetime64[M]")', "floor('MS')", 'floor("MS")', "floor('M')", 'floor("M")', 'Grouper('

/tmp/ipykernel_877/52853772.py:92: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df["ts"] = pd.to_datetime(


AnomalyResult(query_text='Cek anomali mesin XP888A tanggal 1–20 Januari 2025', start=datetime.datetime(2025, 1, 1, 0, 0), end=datetime.datetime(2025, 1, 20, 23, 59, 59), scope='NOT_ALL', machines=['XP888A'], summary=  machine_id  total_points  anomalies            first_ts             last_ts  \
0     XP888A          1714          2 2025-01-01 11:42:57 2025-01-20 23:54:21   

   has_anomaly  
0         True  , details=     machine_id                  ts    value  min_value  max_value  is_low  \
6        XP888A 2025-01-01 13:21:17  11.2243          0         10   False   
1377     XP888A 2025-01-17 04:13:20  19.2340          0         10   False   

      is_high  
6        True  
1377     True  )

Cek anomali mesin XP888A tanggal 1–20 Januari dan februari 2025
Cari di kode:  ["to_period('M')", 'to_period("M")', "astype('datetime64[M]')", 'astype("datetime64[M]")', "floor('MS')", 'floor("MS")', "floor('M')", 'floor("M")', 'Grouper(', "freq='M'", 'freq="M"', "freq='MS'", 'freq="MS"']
Ano

In [59]:
import json
from typing import Any, Dict

def df_summary_to_list(df: pd.DataFrame) -> list[Dict[str, Any]]:
    """
    Convert df_result.summary → list of dict
    Kolom yang diambil: machine_id, total_points, anomalies, first_ts, last_ts, has_anomaly
    Semua timestamp di-cast ke string ISO biar JSON friendly.
    """
    if df is None or len(df) == 0:
        return []

    out = []
    for _, row in df.iterrows():
        out.append({
            "machine_id":    str(row.get("machine_id")),
            "total_points":  int(row.get("total_points", 0)) if not pd.isna(row.get("total_points")) else 0,
            "anomalies":     int(row.get("anomalies", 0)) if not pd.isna(row.get("anomalies")) else 0,
            "first_ts":      row.get("first_ts").strftime("%Y-%m-%d %H:%M:%S") if pd.notna(row.get("first_ts")) else None,
            "last_ts":       row.get("last_ts").strftime("%Y-%m-%d %H:%M:%S") if pd.notna(row.get("last_ts")) else None,
            "has_anomaly":   bool(row.get("has_anomaly")) if not pd.isna(row.get("has_anomaly")) else False,
        })
    return out


def df_details_to_list(df: pd.DataFrame) -> list[Dict[str, Any]]:
    """
    Convert df_result.details → list of dict anomaly events
    Kolom yang diambil:
      machine_id, ts, value, min_value, max_value, is_low, is_high
    """
    if df is None or len(df) == 0:
        return []

    out = []
    for _, row in df.iterrows():
        out.append({
            "machine_id": str(row.get("machine_id")),
            "ts": row.get("ts").strftime("%Y-%m-%d %H:%M:%S") if pd.notna(row.get("ts")) else None,
            "value": float(row.get("value")) if pd.notna(row.get("value")) else None,
            "min_value": float(row.get("min_value")) if pd.notna(row.get("min_value")) else None,
            "max_value": float(row.get("max_value")) if pd.notna(row.get("max_value")) else None,
            "is_low": bool(row.get("is_low")) if not pd.isna(row.get("is_low")) else False,
            "is_high": bool(row.get("is_high")) if not pd.isna(row.get("is_high")) else False,
        })
    return out


def build_data_window_stats(df_ts: pd.DataFrame, sample_dates: int = 5) -> Dict[str, Any]:
    """
    Ambil konteks global data (buat bantu LLM ngerti coverage data kita).
    - ts_min / ts_max
    - total rows
    - beberapa contoh jumlah row / tanggal (bukan semua tanggal supaya tidak bengkak)
    """
    if df_ts.empty:
        return {
            "ts_min": None,
            "ts_max": None,
            "rows_total": 0,
            "rows_per_date_sample": {}
        }

    ts_min = df_ts["ts"].min()
    ts_max = df_ts["ts"].max()
    rows_total = len(df_ts)

    # hitung jumlah row per tanggal
    per_date_counts = (
        df_ts.groupby(df_ts["ts"].dt.date)
             .size()
             .rename("count")
             .sort_index()
    )

    # ambil sample beberapa tanggal “representatif”:
    # - 2 tanggal terawal
    # - 2 tanggal terakhir
    # - 1 tanggal tengah (kalau ada)
    unique_dates = per_date_counts.index.tolist()
    sample_idx = []
    if len(unique_dates) > 0:
        sample_idx.append(0)  # awal
    if len(unique_dates) > 1:
        sample_idx.append(1)  # kedua awal
    if len(unique_dates) > 2:
        sample_idx.append(len(unique_dates)//2)  # tengah
    if len(unique_dates) > 3:
        sample_idx.append(-2)  # kedua terakhir
    if len(unique_dates) > 4:
        sample_idx.append(-1)  # terakhir

    # dedup index terpilih
    sample_idx = [unique_dates[i] for i in dict.fromkeys(sample_idx)]
    rows_per_date_sample = {}
    for d in sample_idx[:sample_dates]:
        rows_per_date_sample[str(d)] = int(per_date_counts.loc[d])

    return {
        "ts_min": ts_min.strftime("%Y-%m-%d %H:%M:%S"),
        "ts_max": ts_max.strftime("%Y-%m-%d %H:%M:%S"),
        "rows_total": int(rows_total),
        "rows_per_date_sample": rows_per_date_sample,
    }


def anomaly_result_to_payload(
    result: AnomalyResult,
    df_ts: pd.DataFrame,
) -> Dict[str, Any]:
    """
    Gabungkan:
    - info query (text, window, mesin)
    - ringkasan per mesin (summary)
    - daftar event anomaly (details)
    - statistik coverage dataset df_ts

    Return dict pure-Python yang aman untuk json.dumps.
    """
    payload = {
        "query_text": result.query_text,
        "resolved_time_window": {
            "start": result.start.strftime("%Y-%m-%d %H:%M:%S"),
            "end":   result.end.strftime("%Y-%m-%d %H:%M:%S"),
        },
        "machines_considered": list(result.machines),
        "scope_label": result.scope,  # "ALL" / "NOT_ALL"
        "data_window_stats": build_data_window_stats(df_ts),
        "summary_per_machine": df_summary_to_list(result.summary),
        "anomaly_events": df_details_to_list(result.details),
    }
    return payload


# OPTIONAL HELPER:
def pretty_print_payload(payload: Dict[str, Any]):
    """
    Hanya buat ngecek outputnya human readable.
    """
    print(json.dumps(payload, indent=2))


In [60]:
OUT_ROOT   = "./runs_anomaly_quad_relclass"

print("\n=== DEMO PREDIKSI ===")
df_thr = pd.DataFrame({
    "machine_id": ["XP888A","PQ667"],
    "max_value": [10, 140],
    "min_value": [0, 110],
})

# df_ts nanti harus sudah bentuk final (machine_id, ts[datetime64], value[float])
df_ts  = load_timeseries_from_df_raw2("/workspace/df_raw2.csv")
examples = [
    "Apakah ada anomali mesin XP888A dan PQ667 7 hari terakhir?",
    "Tolong tampilkan anomali semua mesin bulan ini",
    "Cek anomali mesin XP888A tanggal 1–20 Januari 2025",
    "Cek anomali mesin XP888A tanggal 1–20 Januari dan februari 2025",
    "Cek anomali mesin  bulan Januari dan februari 2025",
    "Cek anomali mesin  bulan Januari 2025",
    "Cek anomali mesin  bulan februari 2025",
    "Ada anomali semua mesin minggu lalu?",
    "Periksa anomali shift malam untuk XP888A",
    "apakah ada anomali kemarin malam?",
    "apakah ada anomali di senin minggu lalu pada pukul 8 pagi hingga 10 pagi?",
    "dalam rentang 30 menit terakhir apakah ada keanehan pada mesin?"
]
for q in examples:
    try:
        # jalankan full pipeline anomaly
        res = run_anomaly_query(q, runner, df_ts=df_ts, df_thr=df_thr)

        # bungkus jadi payload terstruktur
        payload = anomaly_result_to_payload(res, df_ts)

        # debug dump payload yg akan kita kirim ke LLM
        print("\n--- QUERY:", q, "---")
        pretty_print_payload(payload)

    except Exception as e:
        print("\n--- QUERY:", q, "---")
        print("Lewati (error):", e)

print("\nModel tersimpan di:", OUT_ROOT)



=== DEMO PREDIKSI ===

Cari di kode:  ["to_period('M')", 'to_period("M")', "astype('datetime64[M]')", 'astype("datetime64[M]")', "floor('MS')", 'floor("MS")', "floor('M')", 'floor("M")', 'Grouper(', "freq='M'", 'freq="M"', "freq='MS'", 'freq="MS"']

--- QUERY: Apakah ada anomali mesin XP888A dan PQ667 7 hari terakhir? ---
{
  "query_text": "Apakah ada anomali mesin XP888A dan PQ667 7 hari terakhir?",
  "resolved_time_window": {
    "start": "2025-10-17 07:06:03",
    "end": "2025-10-24 07:06:03"
  },
  "machines_considered": [
    "XP888A",
    "PQ667"
  ],
  "scope_label": "NOT_ALL",
  "data_window_stats": {
    "ts_min": "2025-01-01 11:42:57",
    "ts_max": "2025-02-05 11:42:57",
    "rows_total": 6150,
    "rows_per_date_sample": {
      "2025-01-01": 92,
      "2025-01-02": 176,
      "2025-01-19": 176,
      "2025-02-04": 178,
      "2025-02-05": 88
    }
  },
  "summary_per_machine": [],
  "anomaly_events": []
}
Cari di kode:  ["to_period('M')", 'to_period("M")', "astype('dateti

/tmp/ipykernel_877/52853772.py:92: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df["ts"] = pd.to_datetime(


Cari di kode:  ["to_period('M')", 'to_period("M")', "astype('datetime64[M]')", 'astype("datetime64[M]")', "floor('MS')", 'floor("MS")', "floor('M')", 'floor("M")', 'Grouper(', "freq='M'", 'freq="M"', "freq='MS'", 'freq="MS"']

--- QUERY: Cek anomali mesin XP888A tanggal 1–20 Januari 2025 ---
{
  "query_text": "Cek anomali mesin XP888A tanggal 1\u201320 Januari 2025",
  "resolved_time_window": {
    "start": "2025-01-01 00:00:00",
    "end": "2025-01-20 23:59:59"
  },
  "machines_considered": [
    "XP888A"
  ],
  "scope_label": "NOT_ALL",
  "data_window_stats": {
    "ts_min": "2025-01-01 11:42:57",
    "ts_max": "2025-02-05 11:42:57",
    "rows_total": 6150,
    "rows_per_date_sample": {
      "2025-01-01": 92,
      "2025-01-02": 176,
      "2025-01-19": 176,
      "2025-02-04": 178,
      "2025-02-05": 88
    }
  },
  "summary_per_machine": [
    {
      "machine_id": "XP888A",
      "total_points": 1714,
      "anomalies": 2,
      "first_ts": "2025-01-01 11:42:57",
      "last_ts"

In [1]:
# ============================================================
# pipeline_full.py
#
# End-to-end anomaly QA pipeline:
#   1. user query (bahasa natural)
#   2. intent classification (4-head model + smart rules)
#   3. time window resolution
#   4. anomaly scan (bandingkan data ts vs threshold)
#   5. payload siap buat LLM / UI dashboard
#
# Cocok buat:
# - smoke test manual di Colab
# - nanti di-wrap jadi API FastAPI/Flask
# ============================================================

from __future__ import annotations
from typing import Dict, Any, List, Callable, Optional
import os
import torch
import pandas as pd

# -----------------------------
# CONFIG
# -----------------------------
MODEL_ROOT   = "/workspace/runs_anomaly_quad_relclass"  # folder berisi 4 head final_model
TS_CSV_PATH  = "/workspace/df_raw2.csv"        # hasil extractor df_raw2.csv
# TODO prod: load threshold dari Excel pakai loader threshold
DF_THRESHOLDS = pd.DataFrame({
    "machine_id": ["XP888A", "PQ667"],
    "max_value":  [10,       140],
    "min_value":  [0,        110],
})

# -----------------------------
# IMPORT MODUL INTERNAL
# -----------------------------
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

from query_interpret import smart_predict            # post-process intent
from time_parse import resolve_time_window           # text+pred -> TimeWindow(start,end,...)
from anomaly_query_engine import run_anomaly_query   # orchestrator text→AnomalyResult
from other_ts_loader import load_timeseries_from_df_raw2
from payload_formatter import (
    anomaly_result_to_payload,
    pretty_print_payload,
)

# ============================================================
# (1) Runtime model intent (Quad Head)
#    -> diambil dari versi kamu yang asli (:contentReference[oaicite:0]{index=0})
# ============================================================

class AnomalyQuadRunner:
    """
    Wrapper empat head classifier:
      - scope (ALL / NOT_ALL)
      - time_granularity (HOUR / DAY / WEEK / MONTH / SHIFT / YEAR / ...)
      - time_complexity (RANGE_CLEAR / RANGE_REL / PERIOD / ...)
      - case (A1..A15 / R1..R6)
    """
    def __init__(self, scope_dir, gran_dir, cplx_dir, case_dir):
        # Scope
        self.scope_tok = AutoTokenizer.from_pretrained(scope_dir)
        self.scope_cfg = AutoConfig.from_pretrained(scope_dir)
        self.scope_i2l = self.scope_cfg.id2label
        self.scope_m   = AutoModelForSequenceClassification.from_pretrained(scope_dir).eval()

        # Granularity
        self.gran_tok = AutoTokenizer.from_pretrained(gran_dir)
        self.gran_cfg = AutoConfig.from_pretrained(gran_dir)
        self.gran_i2l = self.gran_cfg.id2label
        self.gran_m   = AutoModelForSequenceClassification.from_pretrained(gran_dir).eval()

        # Complexity
        self.cplx_tok = AutoTokenizer.from_pretrained(cplx_dir)
        self.cplx_cfg = AutoConfig.from_pretrained(cplx_dir)
        self.cplx_i2l = self.cplx_cfg.id2label
        self.cplx_m   = AutoModelForSequenceClassification.from_pretrained(cplx_dir).eval()

        # Case
        self.case_tok = AutoTokenizer.from_pretrained(case_dir)
        self.case_cfg = AutoConfig.from_pretrained(case_dir)
        self.case_i2l = self.case_cfg.id2label
        self.case_m   = AutoModelForSequenceClassification.from_pretrained(case_dir).eval()

        # Device
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        for m in [self.scope_m, self.gran_m, self.cplx_m, self.case_m]:
            m.to(self.device)

    @staticmethod
    def _decode(i2l, idx: int):
        # HuggingFace id2label bisa dict {0:'ALL',1:'NOT_ALL',...} atau list
        if isinstance(i2l, dict):
            return i2l.get(idx) or i2l.get(str(idx)) or f"label_{idx}"
        try:
            return i2l[idx]
        except Exception:
            return f"label_{idx}"

    @torch.no_grad()
    def predict(self, text: str, max_len=128):
        # scope
        s = self.scope_tok(
            text,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt"
        ).to(self.device)
        s_logits = self.scope_m(**s).logits.squeeze(0).cpu().numpy()
        s_id = int(s_logits.argmax())
        s_lab = self._decode(self.scope_i2l, s_id)
        s_conf = float(torch.softmax(torch.from_numpy(s_logits), dim=0)[s_id])

        # granularity
        g = self.gran_tok(
            text,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt"
        ).to(self.device)
        g_logits = self.gran_m(**g).logits.squeeze(0).cpu().numpy()
        g_id = int(g_logits.argmax())
        g_lab = self._decode(self.gran_i2l, g_id)
        g_conf = float(torch.softmax(torch.from_numpy(g_logits), dim=0)[g_id])

        # complexity
        x = self.cplx_tok(
            text,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt"
        ).to(self.device)
        x_logits = self.cplx_m(**x).logits.squeeze(0).cpu().numpy()
        x_id = int(x_logits.argmax())
        x_lab = self._decode(self.cplx_i2l, x_id)
        x_conf = float(torch.softmax(torch.from_numpy(x_logits), dim=0)[x_id])

        # case
        c = self.case_tok(
            text,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt"
        ).to(self.device)
        c_logits = self.case_m(**c).logits.squeeze(0).cpu().numpy()
        c_id = int(c_logits.argmax())
        c_lab = self._decode(self.case_i2l, c_id)
        c_conf = float(torch.softmax(torch.from_numpy(c_logits), dim=0)[c_id])

        # route hints awal buat downstream
        route = []
        route.append("aggregate_all_machines" if s_lab == "ALL" else "machine_entity_extractor")
        if g_lab in {"HOUR", "DAY", "WEEK", "MONTH", "YEAR", "SHIFT"}:
            route.append(f"slot_{g_lab.lower()}_extractor")
        if x_lab in {"RANGE_CLEAR", "RANGE_REL", "WEEK_N", "POINT", "PERIOD", "SHIFT"}:
            route.append(f"time_complexity_{x_lab.lower()}")

        return {
            "scope": {"label": s_lab, "confidence": round(s_conf, 4)},
            "time_granularity": {"label": g_lab, "confidence": round(g_conf, 4)},
            "time_complexity": {"label": x_lab, "confidence": round(x_conf, 4)},
            "case": {"label": c_lab, "confidence": round(c_conf, 4)},
            "route_hints": route,
        }


# ============================================================
# (2) Bootstrap util
# ============================================================

def build_runner(model_root: str) -> AnomalyQuadRunner:
    """
    Load semua head classifier dari folder model_root.
    """
    return AnomalyQuadRunner(
        scope_dir=os.path.join(model_root, "scope", "final_model"),
        gran_dir=os.path.join(model_root, "time_granularity", "final_model"),
        cplx_dir=os.path.join(model_root, "time_complexity", "final_model"),
        case_dir=os.path.join(model_root, "case", "final_model"),
    )

def load_sensor_timeseries(ts_csv_path: str) -> pd.DataFrame:
    """
    Load dataframe sensor final (machine_id, ts[datetime64], value[float])
    dari df_raw2.csv yang udah dibersihin sebelumnya.
    """
    # asumsi timestamp historis = MM/DD/YYYY HH:MM
    # kalau ternyata DD/MM/YYYY ubah argumen assume_us_datetime=False
    return load_timeseries_from_df_raw2(
        ts_csv_path,
        assume_us_datetime=True,
        verbose=False,
    )


# ============================================================
# (3) Core: jalankan 1 query user end-to-end
# ============================================================

def run_query_once(
    query_text: str,
    runner: AnomalyQuadRunner,
    df_ts: pd.DataFrame,
    df_thr: pd.DataFrame,
) -> Dict[str, Any]:
    """
    Satu kalimat user → payload anomaly JSON-friendly.

    Step:
    1. smart_predict() → intent final (scope, granularity, complexity, case, route_hints, dll)
    2. resolve_time_window() → TimeWindow(start, end, ...)
    3. run_anomaly_query() → cari anomali di df_ts sesuai window + threshold
    4. anomaly_result_to_payload() → siap kirim ke UI / LLM
    """

    # 1. intent + fix rule (scope SUBSET / ALL, granularity SHIFT/HOUR/WEEK/etc, mapping case A*/R*)
    pred_aug = smart_predict(query_text, runner)

    # 2. interpret waktu (hari ini? minggu lalu? shift malam? 1–20 Jan 2025? dst)
    tw = resolve_time_window(query_text, pred_aug)

    # 3. cari anomaly di window tsb
    #    NOTE: kita pakai versi run_anomaly_query yang SUDAH dipatch
    #    untuk dukung pred_override dan resolve_time_window_fn.
    result = run_anomaly_query(
        text=query_text,
        runner=runner,
        df_ts=df_ts,
        df_thr=df_thr,
        pred_override=pred_aug,
        resolve_time_window_fn=lambda _txt, _pred: tw,
    )

    # 4. bungkus jadi payload json-friendly
    payload = anomaly_result_to_payload(result, df_ts)
    return payload


# ============================================================
# (4) Demo batch (smoke test)
# ============================================================

EXAMPLES = [
    "Apakah ada anomali mesin XP888A dan PQ667 7 hari terakhir?",
    "Tolong tampilkan anomali semua mesin bulan ini",
    "Cek anomali mesin XP888A tanggal 1–20 Januari 2025",
    "Cek anomali mesin XP888A tanggal 1–20 Januari dan februari 2025",
    "Cek anomali semua mesin bulan Januari dan februari 2025",
    "Cek anomali semua mesin bulan Januari 2025",
    "Cek anomali semua mesin bulan februari 2025",
    "Ada anomali semua mesin minggu lalu?",
    "Periksa anomali shift malam untuk XP888A",
    "apakah ada anomali kemarin malam?",
    "apakah ada anomali di senin minggu lalu pada pukul 8 pagi hingga 10 pagi?",
    "dalam rentang 30 menit terakhir apakah ada keanehan pada mesin?",
]

def demo_batch():
    print("=== BOOTSTRAP MODEL & DATA ===")
    runner = build_runner(MODEL_ROOT)
    df_ts  = load_sensor_timeseries(TS_CSV_PATH)
    df_thr = DF_THRESHOLDS.copy()

    print("\n=== DEMO QUERY ===")
    for q in EXAMPLES:
        print("\n-------------------------------------------------")
        print("QUERY:", q)
        try:
            payload = run_query_once(
                query_text=q,
                runner=runner,
                df_ts=df_ts,
                df_thr=df_thr,
            )
            # tampilkan ringkas human-readable
            pretty_print_payload(payload)
        except Exception as e:
            print("ERROR:", e)

    print("\n[INFO]")
    print("Model root      :", MODEL_ROOT)
    print("Sensor rows     :", len(df_ts))
    print("Threshold rows  :", len(df_thr))


# ============================================================
# (5) Entry point
# ============================================================

if __name__ == "__main__":
    demo_batch()


=== BOOTSTRAP MODEL & DATA ===


/workspace/other_ts_loader.py:196: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df["ts"] = pd.to_datetime(



=== DEMO QUERY ===

-------------------------------------------------
QUERY: Apakah ada anomali mesin XP888A dan PQ667 7 hari terakhir?
{
  "query_text": "Apakah ada anomali mesin XP888A dan PQ667 7 hari terakhir?",
  "resolved_time_window": {
    "start": "2025-10-20 07:08:31",
    "end": "2025-10-27 07:08:31"
  },
  "machines_considered": [
    "XP888A",
    "PQ667"
  ],
  "scope_label": "NOT_ALL",
  "data_window_stats": {
    "ts_min": "2025-01-01 11:42:57",
    "ts_max": "2025-02-05 11:42:57",
    "rows_total": 6150,
    "rows_per_date_sample": {
      "2025-01-01": 92,
      "2025-01-02": 176,
      "2025-01-19": 176,
      "2025-02-04": 178,
      "2025-02-05": 88
    }
  },
  "summary_per_machine": [],
  "anomaly_events": []
}

-------------------------------------------------
QUERY: Tolong tampilkan anomali semua mesin bulan ini
{
  "query_text": "Tolong tampilkan anomali semua mesin bulan ini",
  "resolved_time_window": {
    "start": "2025-10-01 00:00:00",
    "end": "2025-1